# Convert to GGUF, then check it

One pass: fetch the checkpoint, work out what architecture it is, convert, quantize, and
verify the result against the original before anything gets published.

The architecture list is read from city96's ComfyUI-GGUF at run time, so a model added
upstream works without editing this notebook. A checkpoint that matches nothing still gets
converted, under a derived template, instead of being forced into somebody else's.

Fill in 1a to 1e, then Runtime → Run all.

**Single-row parameters.** A raw `nn.Parameter` stored as `[1, dim]` - Z-Image's
`cap_pad_token` and `x_pad_token`, Wan's modulation tables - is one row, and nothing
dequantizes a bare Parameter at load time, so it has to stay F32. Upstream's rule tests
`n_dims == 1` and misses it; the rule here is "everything but the last axis is 1".
`llama-quantize` also writes each tensor's dimension count as `ggml_n_dims()`, which drops
trailing 1s, so `[1, dim]` comes back as `[dim]` and ComfyUI refuses it with a size
mismatch - those dimensions are compared against the base file and put back.

**F32 protections and llama.cpp.** `llama-quantize` enforces the per-family skip lists in
city96's `lcpp.patch`, not this notebook's. Anything protected here that comes back
quantized is copied out of the base file in the same pass.

In [ ]:
# @title 1a · What you're doing { display-mode: "form" }

# @markdown Convert, check, or both. `check only` skips straight to verifying whatever is
# @markdown already in the output folder.
JOB = "convert and check"  # @param ["convert and check", "convert only", "check only"]

# @markdown What kind of model this is. Decides which converter runs and which checks apply.
TASK = "image generation"  # @param ["auto-detect", "text / chat LLM", "vision (VQA, captioning, OCR)", "audio-in LLM", "embedding / reranker", "image generation", "video generation"]

# @markdown ---
# @markdown Scratch space for the toolchain, and where the finished files go.
WORK_DIR = "/content/gguf-work"  # @param {type:"string"}
OUT_DIR = "/content/gguf-out"  # @param {type:"string"}

# @markdown Where the result has to load.
RUNTIME = "auto"  # @param ["auto", "llama.cpp / Ollama", "ComfyUI (GGUF nodes)"]

In [ ]:
# @title 1b · The model { display-mode: "form" }

# @markdown Where the checkpoint comes from.
SRC_FROM = "download link"  # @param ["download link", "local path", "Hugging Face repo"]

# @markdown **download link** — the direct URL. Hugging Face is the Download button on the
# @markdown file page, or `https://huggingface.co/<repo>/resolve/main/<file>`. Civitai looks
# @markdown like `https://civitai.com/api/download/models/<id>?fileId=<id>`.
# @markdown Several URLs, comma separated, are fine - extras are treated as companion files.
SRC_URL = ""  # @param {type:"string"}

# @markdown **local path** — a `.safetensors` file, or a model folder for the LLM tasks.
SRC_PATH = ""  # @param {type:"string"}

# @markdown **Hugging Face repo** — for text/vision/audio/embedding, which need the folder.
SRC_REPO = ""  # @param {type:"string"}
SRC_FILE = ""  # @param {type:"string"}
SRC_REVISION = "main"  # @param {type:"string"}

# @markdown ---
# @markdown Leave the tokens blank if you've set Colab secrets called `HF_TOKEN` and
# @markdown `CIVITAI_TOKEN` (the key icon on the left). Civitai needs one for most downloads.
HF_TOKEN = ""  # @param {type:"string"}
CIVITAI_TOKEN = ""  # @param {type:"string"}
# @markdown Parallel connections per download.
CONNECTIONS = 16  # @param {type:"slider", min:1, max:16, step:1}

In [ ]:
# @title 1c · What to build { display-mode: "form" }

# @markdown `mix` keeps whatever precision the checkpoint already uses, which is the safest
# @markdown option and the one everything else is quantized from.
WANT_MIX = True  # @param {type:"boolean"}
WANT_8BIT = True  # @param {type:"boolean"}
WANT_4BIT = True  # @param {type:"boolean"}

RECIPE_8BIT = "Q8_0"  # @param ["Q8_0"]
RECIPE_4BIT = "Q4_K_M"  # @param ["Q4_K_M", "Q4_K_S", "Q4_0", "IQ4_NL", "IQ4_XS"]
# @markdown Anything else, comma separated. `none` to skip.
EXTRA_QUANTS = "none"  # @param ["none", "Q6_K", "Q5_K_M", "Q5_K_S", "Q3_K_M", "Q2_K"] {allow-input: true}

# @markdown ---
# @markdown If a target can't be built the way you asked, `substitute` builds the nearest
# @markdown thing that can be (Q4_K_M becomes Q4_1, Q6_K becomes Q8_0, and so on) rather
# @markdown than leaving you with nothing.
IF_QUANT_UNAVAILABLE = "substitute the nearest"  # @param ["substitute the nearest", "drop it"]

# @markdown ---
# @markdown For multimodal LLMs: also export the `mmproj` projector, which is the image or
# @markdown audio half of the model. `auto` does it for the vision and audio tasks.
EXPECT_MMPROJ = "auto"  # @param ["auto", "yes", "no"]

# @markdown ---
# @markdown What to do about a file that's already in the output folder from an earlier
# @markdown run. `skip` leaves it where it is and only builds what's missing, which is what
# @markdown you want when a run died partway through or you're adding one more quant to a
# @markdown set. Skipped files are still checked. `rebuild` converts everything again and
# @markdown overwrites.
IF_OUTPUT_EXISTS = "skip"  # @param ["skip", "rebuild"]


In [ ]:
# @title 1d · Architecture { display-mode: "form" }

# @markdown `detect` reads the family off the checkpoint's own tensor names. If nothing
# @markdown matches, a template is derived from the weights and the file is converted under
# @markdown a name of its own - it will not be forced into an unrelated family.
# @markdown
# @markdown `force` writes the name below no matter what the weights say. Only do this when
# @markdown you know the checkpoint really is that family and detection is just behind.
ARCH_MODE = "detect"  # @param ["detect", "force"]
# @markdown Name to write. Used by `force`, and as the name for an unrecognised checkpoint
# @markdown under `detect`. Empty means derive one from the filename.
ARCH_NAME = ""  # @param {type:"string"}

# @markdown ---
# @markdown What to do if the checkpoint has already been quantized, i.e. int8 or fp8 weights
# @markdown stored next to their scale tensors.
# @markdown
# @markdown `unpack` rebuilds float weights by multiplying the integers back by their scales,
# @markdown which is the only way to get a usable gguf out of one. Quality stays capped at
# @markdown whatever the original kept. `as-is` writes the packed integers as if they were
# @markdown weights, which produces noise - it's there for inspection, not for shipping.
IF_ALREADY_QUANTIZED = "unpack and convert"  # @param ["unpack and convert", "stop", "convert as-is (unsafe)"]

# @markdown What to do with a target that can't beat the source, like asking for `mix` or
# @markdown `Q8_0` from an 8-bit checkpoint. Those come out bigger with no quality gain.
IF_TARGET_TOO_HIGH = "drop it"  # @param ["drop it", "build it anyway", "stop"]

# @markdown ---
# @markdown `llama-quantize` only knows the architecture names city96's patch adds to it, so
# @markdown a derived one gets rejected with "unknown model architecture".
# @markdown
# @markdown `python` quantizes here instead, applying the protections worked out from this
# @markdown checkpoint. It can't do K-quants (Q4_K_M and friends), only Q8_0, Q5_1, Q5_0,
# @markdown Q4_1 and Q4_0 - anything else is dropped and said so.
# @markdown
# @markdown `stand-in` relabels the file as a known family just long enough to quantize.
# @markdown That gets K-quants working, but llama.cpp then applies that family's hardcoded
# @markdown list of tensors to leave alone, and those names won't match your model.
IF_ARCH_UNKNOWN = "quantize in python"  # @param ["quantize in python", "drop the quants", "stand-in arch"]
# @markdown Which family to borrow for `stand-in`.
# @markdown `auto` takes the highest-scoring family llama-quantize will accept, and
# @markdown never a `shape_fix` family (sd1/sdxl) for a DiT - those rules target 4-D
# @markdown conv tensors.
STAND_IN_ARCH = "auto"  # @param ["auto", "flux", "sd1", "sdxl", "sd3", "aura", "ltxv", "hyvid", "wan", "hidream", "cosmos", "lumina2"]

# @markdown ---
# @markdown How much to print while converting. `every tensor` lists each one with its
# @markdown dtype, shape and why it ended up where it did, which is a lot of output on a big
# @markdown model but the only way to follow a specific layer through. `summary` shows a
# @markdown progress bar plus anything unusual.
PROGRESS = "summary"  # @param ["quiet", "summary", "every tensor"]

# @markdown ---
# @markdown Which ComfyUI-GGUF repo to read architectures from. Forks add their own families,
# @markdown so pointing at one is how a model the base node doesn't know becomes loadable.
# @markdown For Krea 2 that's `RealRebelAI/ComfyUI-GGUF_KREA-2`, which adds `krea2`.
# @markdown Several repos, comma separated, are read as one pool - a fork only adds
# @markdown families, it never overwrites upstream's. Detection sees the union.
GGUF_NODE_REPOS = "city96/ComfyUI-GGUF"  # @param ["city96/ComfyUI-GGUF", "city96/ComfyUI-GGUF, RealRebelAI/ComfyUI-GGUF_KREA-2"] {allow-input: true}
GGUF_NODE_REF = "main"  # @param {type:"string"}
GGUF_NODE_REPO = [r.strip() for r in GGUF_NODE_REPOS.split(",") if r.strip()][0]

# @markdown ---
# @markdown How close a family has to be before its name is written into the file.
# @markdown 1.0 means every one of its detection keys is present, which is what
# @markdown convert.py itself requires. Below the line the checkpoint ships as `pig`
# @markdown instead.
CONFIDENT_AT = 1.0  # @param {type:"slider", min:0.5, max:1.0, step:0.05}

# @markdown `general.architecture = "pig"` makes ComfyUI-GGUF skip its whitelist and run
# @markdown its own `detect_arch()` against the tensor names at load time. Detection then
# @markdown follows whatever version of the node is installed, so a family added upstream
# @markdown later fixes an already-converted file with no reconversion. A made-up literal
# @markdown name is rejected outright and always will be, so `pig` is never worse.
DEFER_WHEN_UNSURE = True  # @param {type:"boolean"}
# @markdown llama.cpp tag to build `llama-quantize` from. Only needed for the 8/4-bit files.
LLAMA_TAG = "b3962"  # @param {type:"string"}

In [ ]:
# @title 1e · Checks { display-mode: "form" }

# @markdown Compare tensor names and shapes against the original checkpoint. This is what
# @markdown finds dropped and mis-shaped weights, so leave it on.
COMPARE_TO_SOURCE = True  # @param {type:"boolean"}
# @markdown Check each quant kept the same tensors and shapes as the file it came from.
COMPARE_QUANTS_TO_BASE = True  # @param {type:"boolean"}
# @markdown Look for NaN and Inf. More tensors is slower, and these files are big.
NAN_CHECK = "normal"  # @param ["off", "quick", "normal", "thorough", "every tensor"]

# @markdown ---
# @markdown Compare the *values* in each gguf against the checkpoint they came from,
# @markdown tensor by tensor, grouped by the part of the network each belongs to. Names
# @markdown and shapes matching proves nothing about whether the numbers survived - a
# @markdown rescaled, transposed or half-converted weight passes every structural check
# @markdown there is. This is how you find which part of the model broke rather than
# @markdown just that it broke. Tensors compared per role; 0 turns it off.
DIFF_SAMPLE = 4  # @param {type:"slider", min:0, max:16, step:1}

# @markdown ---
# @markdown Tie every file produced to the exact checkpoint it came from. The source is
# @markdown fingerprinted, the fingerprint is written into each gguf, and every later pass
# @markdown checks it - so a quant can't be cut from someone else's base file, a stale gguf
# @markdown in the output folder can't be mistaken for this one, and a file that doesn't
# @markdown match what's in 1b fails the checks instead of being shipped.
# @markdown
# @markdown `sampled` reads the header plus evenly spaced chunks, a few seconds even on a
# @markdown large checkpoint. `full` reads every byte, which is exact but slower. A finetune
# @markdown differs in every block, so `sampled` still tells two checkpoints apart.
SOURCE_STAMP = "sampled"  # @param ["sampled", "full", "off"]

# @markdown ---
# @markdown Weights that must stay F32. `auto` takes whatever the detected family
# @markdown protects and adds anything this checkpoint's own tensor names imply on top -
# @markdown a family template can be older than the model in front of it. Type comma
# @markdown separated patterns to override it outright.
KEEP_IN_F32 = "auto"  # @param {type:"string"}

# @markdown ---
# @markdown Files that have to ship next to the gguf. A diffusion gguf is only the
# @markdown transformer, so on its own it won't run.
NEED_VAE = False  # @param {type:"boolean"}
NEED_TEXT_ENCODER = False  # @param {type:"boolean"}
NEED_CONNECTORS = False  # @param {type:"boolean"}
NEED_OTHER_FILES = "none"  # @param {type:"string"}

# @markdown ---
# @markdown `strict` turns warnings into failures.
STRICTNESS = "normal"  # @param ["normal", "strict"]
# @markdown Save the results to `OUT_DIR/verification_report.md`.
SAVE_REPORT = True  # @param {type:"boolean"}
# @markdown Push the files and the report to a Hub repo. Empty keeps everything local.
UPLOAD_REPO = ""  # @param {type:"string"}

### 1f. Apply & check

In [ ]:
import os, re, json, struct, shutil, subprocess, sys, time
from pathlib import Path
from urllib.parse import urlsplit, urlunsplit, parse_qsl, urlencode, unquote

BROWSER_UA = ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
              "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36")

TASKS = {
    "auto-detect":                   ("auto", "auto"),
    "text / chat LLM":               ("llm", "text"),
    "vision (VQA, captioning, OCR)": ("llm", "vision"),
    "audio-in LLM":                  ("llm", "audio"),
    "embedding / reranker":          ("llm", "embedding"),
    "image generation":              ("diffusion", "image"),
    "video generation":              ("diffusion", "video"),
}

DTYPE_BITS = {"F64": 64, "I64": 64, "F32": 32, "I32": 32, "BF16": 16, "F16": 16,
              "I16": 16, "F8_E4M3": 8, "F8_E5M2": 8, "I8": 8, "U8": 8,
              "I4": 4, "U4": 4, "BOOL": 1}

# every suffix a quantizer in the wild has used for "this is the scale for X". The
# narrow version of this list is what let an int8 checkpoint through unrecognised:
# a tensor named `<weight>.scale` matched nothing, SCALES came back empty, the whole
# already-quantized branch never ran, and the packed integers were written out as the
# weights. Detection here has to be wider than the naming convention you expect.
SCALE_PATTERNS = ["weight_scale", "scale_weight", "scale_inv", ".scales", ".scale",
                  "_scale", "absmax", "quant_map", "qweight", "zero_point",
                  "rotation", "hadamard", ".wscale", ".qscale"]

# a real diffusion weight sits in here. Raw int8 codes land around 10-80, which is how
# a missing scale is caught without knowing anything about the architecture.
SANE_STD = (5e-4, 1.0)

# which part of the network a tensor belongs to, first match wins. Used to summarise a
# value comparison by role - one role sitting far worse than the rest says which stage
# of the forward pass is broken, which is more useful than any single tensor name.
ROLES = [
    ("q/k norm",    r"(q_norm|k_norm|norm_q|norm_k|query_norm|key_norm)"),
    ("attention",   r"(attn|attention|qkv|to_q|to_k|to_v|to_out)"),
    ("modulation",  r"(modulation|adaln|ada_ln|scale_shift|\bmod\b|gate)"),
    ("positional",  r"(rope|freqs|pos_embed|position)"),
    ("embedding",   r"(embed|embedder|patch|x_in|img_in|txt_in|cap_|pad_token)"),
    ("output",      r"(final|proj_out|\bhead\b|out_layer|unpatch)"),
    ("feedforward", r"(mlp|ffn|feed_forward|\bw[123]\b|linear[12])"),
    ("norm",        r"(norm|layernorm|rmsnorm|ln_)"),
]

# structural read of a checkpoint, used when no family name fits
BLOCK_HINTS = {
    "double_blocks": "flux-style dual stream",
    "single_blocks": "flux-style single stream",
    "joint_blocks": "sd3-style joint",
    "transformer_blocks": "diffusers-style DiT",
    "input_blocks": "unet (sd1/sdxl era)",
    "output_blocks": "unet (sd1/sdxl era)",
    "middle_block": "unet (sd1/sdxl era)",
    "noise_refiner": "lumina2 / z-image family",
    "context_refiner": "lumina2 / z-image family",
}

PREQUANT_HINTS = ["int8", "int4", "fp8", "f8_", "convrot", "quanto",
                  "nf4", "awq", "gptq", "svdq", "scaled", "nvfp4", "mxfp4"]

# precision-sensitive in any architecture that has them, used when deriving a template
GENERIC_HIPREC = ["modulation", "scale_shift", "pos_embed", "emb_pos", "pos_embedder",
                  "position_embedding", "register", "logit_scale", "freqs"]

FALLBACK_IMG = {"flux", "sd1", "sdxl", "sd3", "aura", "hidream", "cosmos",
                "ltxv", "hyvid", "wan", "lumina2", "qwen_image"}
FALLBACK_TXT = {"t5", "t5encoder", "llama", "qwen2vl", "qwen3", "qwen3vl", "gemma3"}

In [ ]:
# @title listify · turn a comma separated box into a list

def listify(value):
    if not value or value.strip().lower() in ("none", "no", "all", ""):
        return []
    return [v.strip() for v in value.replace(",", " ").split() if v.strip()]

In [ ]:
# @title get_token · form field, then env var, then Colab secret

def get_token(name, given):
    tok = given.strip()
    if tok:
        return tok
    tok = os.environ.get(name, "")
    if tok:
        return tok
    try:
        from google.colab import userdata
        return userdata.get(name) or ""
    except Exception:
        return ""

In [ ]:
MODE, SUBTASK = TASKS[TASK]

DO_CONVERT = JOB in ("convert and check", "convert only")
DO_CHECK = JOB in ("convert and check", "check only")

WORK = Path(WORK_DIR).resolve(); WORK.mkdir(parents=True, exist_ok=True)
OUT = Path(OUT_DIR).resolve(); OUT.mkdir(parents=True, exist_ok=True)
GG = WORK / "ComfyUI-GGUF"
LCPP = WORK / "llama.cpp"
STASH = WORK / "stash"; STASH.mkdir(exist_ok=True)
MODEL_DIR = WORK / "model"; MODEL_DIR.mkdir(exist_ok=True)

HF_TOK = get_token("HF_TOKEN", HF_TOKEN)
CIV_TOK = get_token("CIVITAI_TOKEN", CIVITAI_TOKEN)

URLS = [u for u in re.split(r"[,\s]+", SRC_URL.strip()) if u]
if DO_CONVERT and SRC_FROM == "download link" and not URLS:
    raise ValueError("SRC_FROM is 'download link' but SRC_URL is empty - fill it in in 1b")
if DO_CONVERT and SRC_FROM == "local path" and not SRC_PATH.strip():
    raise ValueError("SRC_FROM is 'local path' but SRC_PATH is empty - fill it in in 1b")
if DO_CONVERT and SRC_FROM == "Hugging Face repo" and not SRC_REPO.strip():
    raise ValueError("SRC_FROM is 'Hugging Face repo' but SRC_REPO is empty - fill it in in 1b")

if MODE == "auto":
    single = (SRC_FILE.strip().endswith(".safetensors")
              or SRC_PATH.strip().endswith(".safetensors")
              or any(u.endswith(".safetensors") for u in URLS))
    MODE = "diffusion" if single else "llm"
    print("auto-detect -> pipeline:", MODE)

TARGET = RUNTIME
if TARGET == "auto":
    TARGET = "ComfyUI (GGUF nodes)" if MODE == "diffusion" else "llama.cpp / Ollama"

QUANTS = [q for on, q in [(WANT_8BIT, RECIPE_8BIT), (WANT_4BIT, RECIPE_4BIT)] if on]
QUANTS += [q.upper() for q in listify(EXTRA_QUANTS)]
QUANTS = list(dict.fromkeys(QUANTS))
EXPECT = (["mix"] if WANT_MIX else []) + QUANTS
if DO_CONVERT and not EXPECT:
    raise ValueError("nothing ticked in 1c - there'd be nothing to build")

WANT_MMPROJ = {"auto": SUBTASK in ("vision", "audio"), "yes": True, "no": False}[EXPECT_MMPROJ]
NEED_LLAMA = bool(QUANTS) or MODE == "llm"

COMPANIONS = []
if NEED_VAE:
    COMPANIONS.append(("VAE", ["*vae*"]))
if NEED_TEXT_ENCODER:
    COMPANIONS.append(("text encoder", ["*text_encoder*", "*t5*", "*gemma*", "*clip*", "*umt5*"]))
if NEED_CONNECTORS:
    COMPANIONS.append(("connectors", ["*connector*"]))
for pat in listify(NEED_OTHER_FILES):
    COMPANIONS.append((pat, [pat]))

F32_OVERRIDE = None if KEEP_IN_F32.strip().lower() == "auto" else \
    [p.strip() for p in KEEP_IN_F32.split(",") if p.strip()]
F32_PATTERNS = F32_OVERRIDE or []

NAN_SAMPLE = {"off": None, "quick": 20, "normal": 40, "thorough": 200, "every tensor": 0}[NAN_CHECK]
WANT_5D = SUBTASK == "video"
STRICT = STRICTNESS == "strict"

print("job      :", JOB)
print("pipeline :", MODE, "/", SUBTASK)
print("runtime  :", TARGET)
print("building :", ", ".join(EXPECT) + (" + mmproj" if WANT_MMPROJ else ""))
print("arch     :", ARCH_MODE + (f" -> {ARCH_NAME}" if ARCH_NAME.strip() else ""))
print("out      :", OUT)
print("tokens   : hf=" + ("set" if HF_TOK else "none") + "  civitai=" + ("set" if CIV_TOK else "none"))

## Setup

In [ ]:
!apt-get -qq install -y aria2 > /dev/null 2>&1
!pip install -q "gguf>=0.13" safetensors requests "huggingface_hub>=0.26" torch tqdm

import ast, requests
import numpy as np
import gguf

HAVE_ARIA2 = shutil.which("aria2c") is not None
RAW = (f"https://raw.githubusercontent.com/{GGUF_NODE_REPO.strip()}"
       f"/{GGUF_NODE_REF.strip() or 'main'}")
print("aria2c:", "ready" if HAVE_ARIA2 else "missing, will fall back to single stream")

## Download

In [ ]:
# @title add_query_param · add or replace one parameter on a URL

def add_query_param(url, key, value):
    parts = urlsplit(url)
    query = dict(parse_qsl(parts.query, keep_blank_values=True))
    query[key] = value
    return urlunsplit(parts._replace(query=urlencode(query)))

In [ ]:
# @title auth_for · attach the right credential for the host

def auth_for(url):
    """Hugging Face wants an Authorization header. Civitai wants a token query parameter,
    because its download URL redirects to a pre-signed CDN link and a second credential on
    that request gets rejected."""
    headers = {"User-Agent": BROWSER_UA}
    host = urlsplit(url).netloc.lower()

    if "huggingface.co" in host or "hf.co" in host:
        if HF_TOK:
            headers["Authorization"] = f"Bearer {HF_TOK}"
    elif "civitai" in host:
        if CIV_TOK:
            url = add_query_param(url, "token", CIV_TOK)

    return url, headers

In [ ]:
# @title filename_for · ask the server what the file is actually called

def filename_for(url, headers, override=""):
    if override.strip():
        return override.strip()

    try:
        r = requests.get(url, headers={**headers, "Range": "bytes=0-0"},
                         stream=True, timeout=60, allow_redirects=True)
        cd = r.headers.get("content-disposition", "")
        r.close()
        m = re.search(r"filename\*=UTF-8''([^;]+)", cd) or re.search(r'filename="?([^";]+)', cd)
        if m:
            return unquote(m.group(1)).strip()
    except Exception as e:
        print("  couldn't ask the server for a name:", e)

    return unquote(Path(urlsplit(url).path).name) or "download.bin"

In [ ]:
# @title download · one URL to disk, skipping anything already there

def download(url, dest_dir, override=""):
    dest_dir = Path(dest_dir); dest_dir.mkdir(parents=True, exist_ok=True)
    url, headers = auth_for(url)
    name = filename_for(url, headers, override)
    dest = dest_dir / name

    # aria2 leaves a .aria2 control file next to a transfer it hasn't finished. Without
    # this the next run sees a big file, calls it done, and converts half a checkpoint.
    control = dest.with_name(dest.name + ".aria2")
    if dest.exists() and dest.stat().st_size > 0 and not control.exists():
        print(f"  have {name} ({dest.stat().st_size / 2**30:.2f} GB)")
        return dest
    if control.exists():
        print(f"  {name} was left part-finished, resuming")

    print(f"  {name}")
    if HAVE_ARIA2:
        command = [
            "aria2c", "--console-log-level=warn",
            "-c", "-x", str(CONNECTIONS), "-s", str(CONNECTIONS), "-k", "1M",
            "--max-tries=5", "--retry-wait=5",
            "--allow-overwrite=true", "--auto-file-renaming=false",
            "--summary-interval=5", f"--user-agent={BROWSER_UA}",
            *[f"--header={k}: {v}" for k, v in headers.items() if k != "User-Agent"],
            url, "-d", str(dest_dir), "-o", name,
        ]
        proc = subprocess.Popen(command, stdout=subprocess.PIPE,
                                stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in proc.stdout:
            if line.strip():
                print("   ", line.rstrip())
        if proc.wait() != 0:
            raise RuntimeError(f"aria2c exited {proc.returncode} on {name}")
    else:
        with requests.get(url, headers=headers, stream=True, timeout=120,
                          allow_redirects=True) as r:
            r.raise_for_status()
            with open(dest, "wb") as fh:
                for chunk in r.iter_content(1 << 22):
                    fh.write(chunk)

    print(f"  done, {dest.stat().st_size / 2**30:.2f} GB")
    return dest

In [ ]:
SRC = None
COMPANION_FILES = []

if SRC_FROM == "download link" and URLS:
    got = [download(u, MODEL_DIR) for u in URLS]
    SRC = got[0]
    COMPANION_FILES = got[1:]
    for extra in COMPANION_FILES:
        shutil.copy2(extra, OUT / extra.name)

elif SRC_FROM == "local path" and SRC_PATH.strip():
    SRC = Path(SRC_PATH).resolve()
    if not SRC.exists():
        raise FileNotFoundError(SRC)

elif SRC_FROM == "Hugging Face repo" and SRC_REPO.strip():
    from huggingface_hub import snapshot_download, hf_hub_download
    if SRC_FILE.strip():
        SRC = Path(hf_hub_download(SRC_REPO.strip(), SRC_FILE.strip(),
                                   revision=SRC_REVISION.strip() or "main",
                                   local_dir=str(MODEL_DIR), token=HF_TOK or None))
    else:
        SRC = Path(snapshot_download(SRC_REPO.strip(),
                                     revision=SRC_REVISION.strip() or "main",
                                     local_dir=str(MODEL_DIR), token=HF_TOK or None))

print("\nsource:", SRC or "none (check only)")
if COMPANION_FILES:
    print("companions:", ", ".join(p.name for p in COMPANION_FILES))

## The checkpoint

In [ ]:
# @title read_header · tensor list from a safetensors file, no weights loaded

def read_header(path):
    with open(path, "rb") as fh:
        (n,) = struct.unpack("<Q", fh.read(8))
        return {k: v for k, v in json.loads(fh.read(n)).items() if k != "__metadata__"}

In [ ]:
# @title read_field · one metadata value out of a gguf, safe on numpy 2

def read_field(reader, key):
    """gguf-py hands back size-1 arrays and numpy 2 refuses to int() those, which is why
    upstream's fix_5d_tensors.py dies with 'only 0-dimensional arrays can be converted
    to Python scalars'."""
    field = reader.get_field(key) if hasattr(reader, "get_field") else reader.fields.get(key)
    if field is None:
        return None

    part = field.parts[field.data[-1]]
    if field.types and field.types[0] == gguf.GGUFValueType.STRING:
        return bytes(part).decode("utf-8", errors="replace")
    return part.tolist()[0] if hasattr(part, "tolist") else part

In [ ]:
# @title is_complete · did the file finish being written

def safetensors_is_complete(path):
    """The header says where every tensor ends, so the file has a size it must be. A
    download or copy that was cut short is shorter than that, and read_header can't see it
    because it only reads the front."""
    path = Path(path)
    with open(path, "rb") as fh:
        (n,) = struct.unpack("<Q", fh.read(8))
        header = json.loads(fh.read(n))

    end = 0
    for key, meta in header.items():
        if key != "__metadata__":
            end = max(end, int(meta["data_offsets"][1]))

    need = 8 + n + end
    have = path.stat().st_size
    return have >= need, need, have


def gguf_is_complete(path):
    """A run killed part way through write_tensors_to_file leaves a gguf whose header
    promises more data than the file holds. gguf-py mmaps it and hands back short reads
    rather than complaining, so the size is checked directly.

    data_offset is absolute in current gguf-py and was relative in older ones, so both
    readings are taken and the larger requirement wins."""
    path = Path(path)
    have = path.stat().st_size
    try:
        reader = gguf.GGUFReader(str(path))
        need = max([int(t.data_offset) + int(t.n_bytes) for t in reader.tensors] or [0])
        need = max(need, sum(int(t.n_bytes) for t in reader.tensors))
        del reader
    except Exception as e:
        return False, f"it can't be opened as a gguf at all ({type(e).__name__})"

    if have >= need:
        return True, f"{have / 2**30:.2f} GB, all its header describes"
    return False, (f"{have / 2**30:.2f} GB of the {need / 2**30:.2f} GB its header "
                   f"describes")


In [ ]:
# @title source_fingerprint · a short id for the checkpoint in 1b

def file_digest(path, method):
    """`full` reads every byte. `sampled` reads the front, the back and evenly spaced
    chunks in between, which is seconds rather than minutes on a large checkpoint and still
    separates two models - a finetune differs in every block, not in one place."""
    import hashlib

    path = Path(path)
    size = path.stat().st_size
    h = hashlib.sha256()
    # deliberately not the filename: the same checkpoint arrives under different names
    # from Civitai, HF and a local copy, and all three should fingerprint the same
    h.update(f"{size}:".encode())

    with open(path, "rb") as fh:
        if method == "full":
            for chunk in iter(lambda: fh.read(1 << 24), b""):
                h.update(chunk)
        else:
            span = 1 << 22                       # 4 MB per probe
            step = max(span, size // 48)
            for pos in range(0, size, step):
                fh.seek(pos)
                h.update(fh.read(span))
            fh.seek(max(0, size - span))         # always include the tail
            h.update(fh.read(span))

    return h.hexdigest()


def source_fingerprint(source, method):
    """One id over every safetensors file in the source, so a multi-file model can't be
    confused with a subset of itself."""
    if method == "off" or source is None:
        return {}

    import hashlib

    source = Path(source)
    files = sorted(source.rglob("*.safetensors")) if source.is_dir() else [source]
    if not files:
        return {}

    size = sum(f.stat().st_size for f in files)
    if len(files) == 1:
        digest = file_digest(files[0], method)
    else:
        roll = hashlib.sha256()
        for f in files:
            roll.update(file_digest(f, method).encode())
        digest = roll.hexdigest()

    return {"file": source.name, "size": size, "hash": digest, "method": method}


def write_stamp(writer):
    """Written into every file produced. This is what later passes compare against."""
    if not SRC_FP:
        return
    writer.add_string("general.source.file", str(SRC_FP["file"]))
    writer.add_string("general.source.hash", str(SRC_FP["hash"]))
    writer.add_string("general.source.hash_method", str(SRC_FP["method"]))
    writer.add_uint64("general.source.size", int(SRC_FP["size"]))


def read_stamp(path):
    reader = gguf.GGUFReader(str(path))
    stamp = {name: read_field(reader, f"general.source.{name}")
             for name in ("file", "size", "hash", "hash_method")}
    del reader
    return {k: v for k, v in stamp.items() if v is not None}


def stamp_matches(path):
    """`match`, `different`, `unstamped` or `unchecked`, with something to print."""
    if not SRC_FP:
        return "unchecked", "SOURCE_STAMP is off, nothing ties this file to a checkpoint"

    stamp = read_stamp(path)
    if not stamp.get("hash"):
        return "unstamped", ("no source stamp - made before this check existed, or by "
                             "another tool")
    if stamp.get("hash_method") != SRC_FP["method"]:
        return "unchecked", (f"stamped with the {stamp.get('hash_method')!r} method, this "
                             f"run is using {SRC_FP['method']!r} - not comparable")
    if stamp["hash"] == SRC_FP["hash"]:
        return "match", f"{stamp.get('file')} {stamp['hash'][:12]}"

    return "different", (f"stamped {stamp.get('file')} {stamp['hash'][:12]}, the checkpoint "
                         f"in 1b is {SRC_FP['file']} {SRC_FP['hash'][:12]}")


In [ ]:
# @title strip_prefix · what the converter keeps, and what it quietly throws away

def strip_prefix(keys):
    prefix = None
    for pfx in ("model.diffusion_model.", "model."):
        if any(k.startswith(pfx) for k in keys):
            prefix = pfx
            break
    if prefix is None and keys and all(k.startswith("net.") for k in keys):
        prefix = "net."
    if prefix is None:
        return {k: k for k in keys}, []

    kept, dropped = {}, []
    for k in keys:
        if prefix not in k:
            dropped.append(k)
        else:
            kept[k] = k.replace(prefix, "")
    return kept, dropped

In [ ]:
# @title find_scale_pairs · match each packed weight to its scale tensor

def find_scale_pairs(names):
    """Returns {weight: scale} for the layouts that can be undone with a multiply, plus
    anything packed in a way that can't be (GPTQ/AWQ store 4-bit weights bit-packed into
    int32 with group indices, which needs the original packing code to reverse)."""
    names = set(names)
    pairs = {}
    packed = sorted(n for n in names
                    if any(t in n.lower() for t in ("qweight", "qzeros", "g_idx", "quant_map")))

    # longest first, so ".scale_inv" is not eaten by ".scale"
    suffixes = ("_scale_inv", ".scale_inv", "_weight_scale", ".weight_scale",
                "_scale", ".scales", ".scale", ".absmax", ".wscale", ".qscale")

    for key in names:
        weight = None
        for suffix in suffixes:
            if key.endswith(suffix) and len(key) > len(suffix):
                stem = key[: -len(suffix)]
                # `blk.q.weight.scale` -> `blk.q.weight`, `blk.q.scale` -> `blk.q.weight`
                for cand in (stem, stem + ".weight"):
                    if cand in names and cand != key:
                        weight = cand
                        break
            if weight:
                break
        if key.endswith(".scale_weight"):
            cand = key[: -len(".scale_weight")] + ".weight"
            weight = cand if cand in names else weight

        if weight:
            pairs[weight] = key

    return pairs, packed

In [ ]:
SRC_SHAPES = None
SRC_FP = {}
SRC_LABEL = str(SRC) if SRC else "not given"

if SRC is not None:
    files = sorted(SRC.rglob("*.safetensors")) if SRC.is_dir() else [SRC]

    # before anything is read as weights: a checkpoint that stopped downloading part way
    # parses fine at the header and turns into noise at the far end of the file
    for f in files:
        ok, need, have = safetensors_is_complete(f)
        if not ok:
            raise RuntimeError(
                f"{f.name} is {have / 2**30:.2f} GB but its header describes "
                f"{need / 2**30:.2f} GB - the download or copy didn't finish. Delete it "
                f"and run 1b again rather than converting a truncated checkpoint.")
    if files:
        print(f"{len(files)} file(s), all the length their headers say they should be")

    if files:
        SRC_SHAPES = {}
        for f in files:
            SRC_SHAPES.update(read_header(f))
        print(f"{len(SRC_SHAPES)} tensors across {len(files)} file(s)")

    SRC_FP = source_fingerprint(SRC, SOURCE_STAMP)
    if SRC_FP:
        print(f"source id: {SRC_FP['hash'][:16]} ({SRC_FP['method']}, "
              f"{SRC_FP['size'] / 2**30:.2f} GB) - written into every file produced")
    else:
        print("[!] SOURCE_STAMP is off, so nothing produced here will record which "
              "checkpoint it came from")

if SRC_SHAPES:
    dtypes = {}
    for v in SRC_SHAPES.values():
        dtypes[v["dtype"]] = dtypes.get(v["dtype"], 0) + 1
    print("dtypes:", ", ".join(f"{k}={v}" for k, v in sorted(dtypes.items(), key=lambda x: -x[1])))

    over_4d = [k for k, v in SRC_SHAPES.items() if len(v["shape"]) > 4]
    if over_4d:
        print(f"over 4 dimensions: {len(over_4d)} -> these get stashed and restored after quantizing")
        for k in over_4d[:4]:
            print(f"   {k}  {SRC_SHAPES[k]['shape']}")

    # pair by name first - that is the only evidence that actually matters - and fall
    # back to the pattern list for anything that looks like a scale but pairs with
    # nothing, so an unknown layout still shows up in the report instead of silently
    # being treated as a weight
    SRC_PAIRS, SRC_PACKED = find_scale_pairs(SRC_SHAPES)
    SCALES = sorted(set(SRC_PAIRS.values()) | {
        k for k in SRC_SHAPES
        if any(p in k.lower() for p in SCALE_PATTERNS)
        and "scale_shift" not in k.lower()})

    # the scale tensors are F32 and there's one per weight, so counting everything would
    # call an int8 checkpoint F32 on a coin flip
    wd = {}
    for k, v in SRC_SHAPES.items():
        if k in SCALES or len(v["shape"]) < 2:
            continue
        wd[v["dtype"]] = wd.get(v["dtype"], 0) + 1
    main_dtype = max(wd, key=wd.get) if wd else None
    SRC_BITS = DTYPE_BITS.get(main_dtype)
    print("weight dtypes:", ", ".join(f"{k}={v}" for k, v in
                                      sorted(wd.items(), key=lambda x: -x[1])) or "none")

    if SCALES:
        print(f"\n[!] {len(SCALES)} quantization scale tensor(s), {len(SRC_PAIRS)} of "
              f"which pair with a weight by name, e.g. {SCALES[:2]}")
        print("    This checkpoint was already quantized. The scales have to be applied")
        print("    before anything is written out, or the file is noise.")

    # sub-16-bit weights that pair with nothing are the dangerous case: there is no
    # scale to apply and no way to know it is missing except by counting
    orphan_q = [k for k, v in SRC_SHAPES.items()
                if v["dtype"] in {"I8", "U8", "I4", "U4", "F8_E4M3", "F8_E5M2"}
                and len(v["shape"]) >= 2 and k not in SRC_PAIRS and k not in SCALES]
    if orphan_q:
        print(f"[!] {len(orphan_q)} sub-16-bit weight(s) with no scale anywhere in the "
              f"file, e.g. {orphan_q[:2]}")
        print("    fp8 dtypes are self-describing and fine. Integer dtypes are not - if")
        print("    these are I8/I4 the scales live outside this file and it cannot be")
        print("    converted. Get the float checkpoint.")
    if SRC_BITS and SRC_BITS < 16:
        print(f"[!] {main_dtype} is {SRC_BITS} bits per weight - already quantized once.")
    hits = [h for h in PREQUANT_HINTS if h in Path(SRC_LABEL).name.lower()]
    if hits:
        print(f"[!] filename says {', '.join(hits)}")
else:
    SCALES, SRC_BITS, main_dtype = [], None, None
    SRC_PAIRS, SRC_PACKED, orphan_q = {}, [], []

## Architecture

Which families exist, how each is recognised, and which of its weights must stay F32 all
live in city96's ComfyUI-GGUF. These are read from that repo at run time rather than kept
as a copy here.

In [ ]:
# @title parse_model_templates · pull every architecture out of convert.py

def parse_model_templates(source):
    """Read the ModelXxx classes with ast. Nothing fetched is executed."""
    wanted = ("arch", "keys_detect", "keys_hiprec", "keys_banned", "keys_ignore", "shape_fix")
    by_class = {}

    for node in ast.walk(ast.parse(source)):
        if not isinstance(node, ast.ClassDef):
            continue
        info = {"bases": [b.id for b in node.bases if isinstance(b, ast.Name)]}
        for stmt in node.body:
            if (isinstance(stmt, ast.Assign) and len(stmt.targets) == 1
                    and isinstance(stmt.targets[0], ast.Name)
                    and stmt.targets[0].id in wanted):
                try:
                    info[stmt.targets[0].id] = ast.literal_eval(stmt.value)
                except Exception:
                    pass
        by_class[node.name] = info

    def resolve(name, key, seen=()):
        info = by_class.get(name, {})
        if key in info:
            return info[key]
        for base in info.get("bases", []):
            if base not in seen:
                got = resolve(base, key, seen + (name,))
                if got is not None:
                    return got
        return None

    # convert.py tries families in the order of its arch_list and takes the first match,
    # and that order isn't the class declaration order
    order = []
    for node in ast.parse(source).body:
        if (isinstance(node, ast.Assign) and len(node.targets) == 1
                and getattr(node.targets[0], "id", None) == "arch_list"):
            order = [e.id for e in node.value.elts if isinstance(e, ast.Name)]

    out = {}
    for name in sorted(by_class, key=lambda n: order.index(n) if n in order else len(order)):
        arch = resolve(name, "arch")
        if not arch or arch == "invalid":
            continue
        out[arch] = {
            "class": name,
            "keys_detect": resolve(name, "keys_detect") or [],
            "keys_hiprec": resolve(name, "keys_hiprec") or [],
            "keys_banned": resolve(name, "keys_banned") or [],
            "shape_fix": bool(resolve(name, "shape_fix")),
        }
    return out

In [ ]:
# @title parse_arch_lists · pull the loader's whitelists out of loader.py

def parse_arch_lists(source):
    out = {}
    for node in ast.parse(source).body:
        if (isinstance(node, ast.Assign) and len(node.targets) == 1
                and isinstance(node.targets[0], ast.Name)
                and "ARCH" in node.targets[0].id.upper()):
            try:
                out[node.targets[0].id] = set(ast.literal_eval(node.value))
            except Exception:
                pass
    return out

In [ ]:
# @title llama_quantize_archs · which names the patched llama.cpp will accept

def llama_quantize_archs():
    """city96's lcpp.patch adds a fixed set of image architectures to llama.cpp's enum.
    Anything outside it is refused with "unknown model architecture". Read from the patch
    so a name added upstream is picked up without editing this."""
    try:
        patch = requests.get(f"{RAW}/tools/lcpp.patch", timeout=60).text
        # only lines the patch adds - the diff context has llama.cpp's own archs in it
        names = set(re.findall(r'^\+\s*\{\s*LLM_ARCH_\w+,\s*"([a-z0-9_]+)"',
                               patch, re.M))
        if names:
            return names
    except Exception as e:
        print("  couldn't read lcpp.patch:", e)

    return {"flux", "sd1", "sdxl", "sd3", "aura", "ltxv", "hyvid", "wan",
            "hidream", "cosmos", "lumina2"}

In [ ]:
# @title score_families · how close is each architecture, not just which one matches

def score_families(names, templates):
    """convert.py demands every key of some keys_detect group. One renamed block in a
    merge and a checkpoint falls off a cliff from a real family to nothing at all, which
    is the moment a name gets invented and the file stops loading.

    Scoring keeps the same 1.0 bar for writing a literal name and gives everything else
    a position instead of a silence.

      1.0   every key of some group present, nothing banned - what convert.py requires
      <1.0  best group partially present: a rename, a dropped head, an added adapter
      0.0   a banned key is present, or nothing in common
    """
    names = set(names)
    ranked = []

    for arch, t in templates.items():
        banned = [k for k in t["keys_banned"] if k in names]
        if banned:
            ranked.append((arch, 0.0, f"banned key present: {banned[0]}"))
            continue

        groups = [g for g in t["keys_detect"] if g]
        if not groups:
            continue

        best, hit, total = 0.0, 0, 0
        for g in groups:
            present = sum(1 for k in g if k in names)
            if present / len(g) > best:
                best, hit, total = present / len(g), present, len(g)

        # weak corroboration: a finetune keeps the family's precision-sensitive weights
        # even when a detect key gets renamed
        corr = sum(1 for p in t["keys_hiprec"] if any(p in n for n in names))
        if corr and best < 1.0:
            best = min(0.99, best + 0.05 * corr)

        why = f"{hit}/{total} detect keys" + (f", {corr} hiprec" if corr else "")
        ranked.append((arch, round(best, 3), why))

    ranked.sort(key=lambda r: -r[1])
    return ranked


def fingerprint(shapes):
    """Structural read, independent of any template. This is what says 'a DiT shaped
    like flux' when no name fits, and it is what picks a safe stand-in."""
    names = list(shapes)
    hints = sorted({v for k, v in BLOCK_HINTS.items() if any(k in n for n in names)})

    blocks = {}
    for n in names:
        m = re.search(r"(?:^|\.)((?:[a-z_]+_)?blocks|layers)\.(\d+)\.", n)
        if m:
            blocks[m.group(1)] = max(blocks.get(m.group(1), 0), int(m.group(2)) + 1)

    dims = {}
    for n, v in shapes.items():
        if len(v["shape"]) == 2 and n.endswith(".weight"):
            dims[v["shape"][-1]] = dims.get(v["shape"][-1], 0) + 1

    conv = sum(1 for v in shapes.values() if len(v["shape"]) == 4)
    return {
        "hints": hints,
        "blocks": blocks,
        "hidden": max(dims, key=dims.get) if dims else None,
        "conv4d": conv,
        "nd5": sum(1 for v in shapes.values() if len(v["shape"]) > 4),
        "params": sum(int(np.prod(v["shape"])) for v in shapes.values()),
        "is_dit": conv < 20 and bool(blocks),
    }


def pick_quant_arch(ranked, llama_ok, templates, fp):
    """Which name llama-quantize is told. It only accepts what lcpp.patch adds, and it
    applies that family's hardcoded skip list - which restore_from_base then repairs.
    Never a shape_fix family for a DiT: those rules are aimed at 4-D conv tensors."""
    if STAND_IN_ARCH != "auto" and STAND_IN_ARCH in llama_ok:
        return STAND_IN_ARCH, "set by hand in 1d"
    for arch, score, _ in ranked:
        if score <= 0 or arch not in llama_ok:
            continue
        if fp["is_dit"] and templates.get(arch, {}).get("shape_fix"):
            continue
        return arch, f"nearest family llama-quantize accepts (score {score})"
    for cand in ("flux", "lumina2", "sd3"):
        if cand in llama_ok and not templates.get(cand, {}).get("shape_fix"):
            return cand, "nothing scored - falling back to a plain DiT skip list"
    return (sorted(llama_ok)[0] if llama_ok else "flux"), "last resort"


def load_templates(repos, ref="main"):
    """Union the tables from every repo given. Earlier repos win ties, so upstream stays
    authoritative and a fork can only add families."""
    merged, img, txt, seen = {}, set(), set(), []

    for repo in [r.strip() for r in re.split(r"[,\s]+", repos) if r.strip()]:
        raw = f"https://raw.githubusercontent.com/{repo}/{ref}"
        try:
            tpl = parse_model_templates(requests.get(f"{raw}/tools/convert.py",
                                                     timeout=60).text)
            lists = parse_arch_lists(requests.get(f"{raw}/loader.py", timeout=60).text)
        except Exception as e:
            print(f"  [!] {repo}: {e}")
            continue
        for arch, t in tpl.items():
            merged.setdefault(arch, dict(t, repo=repo))
        img |= lists.get("IMG_ARCH_LIST", set())
        txt |= lists.get("TXT_ARCH_LIST", set())
        seen.append(f"{repo}@{ref} ({len(tpl)})")

    return merged, img, txt, ", ".join(seen)


In [ ]:
# @title derive_hiprec · work out which weights need F32 without a template

def derive_hiprec(shapes):
    """A raw nn.Parameter shows up as a key that doesn't end in .weight or .bias, and those
    are exactly what the upstream keys_hiprec lists protect - modulation tables, scale-shift
    tables, positional embeddings, learnable registers. One-dimensional and very small
    tensors are already forced to F32 by the converter itself.

    The reason a raw Parameter needs protecting is that nothing dequantizes it at runtime:
    ComfyUI-GGUF patches nn.Linear, nn.Conv2d, nn.Embedding and the norms, and a bare
    Parameter hanging off the module is none of those. A `.weight` under a real module is
    handled, so a bare substring like "modulation" must not be allowed to drag one in -
    Wan's `blocks.0.modulation` is a Parameter and needs F32, Z-Image's
    `layers.0.adaLN_modulation.0.weight` is an nn.Linear and does not. A pattern that names
    a `.weight` outright, like HiDream's `.ff_i.gate.weight`, still matches: that one is a
    deliberate upstream choice rather than a guess.

    Names come back stripped. The checkpoint's keys may carry a `model.diffusion_model.`
    prefix that the converter removes, and every consumer of this list - convert_diffusion,
    quantize_python, check_dtypes - matches against the stripped name. A prefixed entry is
    longer than the name it is meant to match, so it silently matches nothing and the
    protection quietly does not happen."""
    kept, _dropped = strip_prefix(list(shapes))
    name_of = lambda key: kept.get(key, key)
    found = set()

    for key, meta in shapes.items():
        if len(meta["shape"]) < 2 or key in SCALES:
            continue
        if key.rsplit(".", 1)[-1] not in ("weight", "bias"):
            found.add(name_of(key))

    patterns = sorted({p for t in TEMPLATES.values() for p in t["keys_hiprec"]})
    patterns += [p for p in GENERIC_HIPREC if p not in patterns]
    for key in shapes:
        if key in SCALES:
            continue
        module_weight = key.rsplit(".", 1)[-1] in ("weight", "bias")
        for p in patterns:
            # a loose pattern is aimed at raw Parameters only; it doesn't get to claim a
            # weight or bias that belongs to a module the loader already handles
            if module_weight and not p.endswith((".weight", ".bias")):
                continue
            if p in key:
                found.add(name_of(key))
                break

    return sorted(found), patterns

In [ ]:
print("reading architecture definitions")
TEMPLATES, IMG_ARCH, TXT_ARCH, ARCH_SOURCE = load_templates(GGUF_NODE_REPOS, GGUF_NODE_REF)
if not TEMPLATES:
    TEMPLATES, IMG_ARCH, TXT_ARCH = {}, FALLBACK_IMG, FALLBACK_TXT
    ARCH_SOURCE = "built-in fallback"

print(f"  {len(TEMPLATES)} families from {ARCH_SOURCE}")
print(f"  loadable: {', '.join(sorted(IMG_ARCH))}")
gap = sorted(set(TEMPLATES) - IMG_ARCH)
if gap:
    print(f"  convertible but on no loader list: {', '.join(gap)}")

DETECTED, DERIVED, ARCH_SCORES, FP = [], False, [], {}
SHIP_ARCH = QUANT_ARCH = ARCH = ARCH_NAME.strip()

if SRC_SHAPES and MODE == "diffusion":
    try:
        LLAMA_ARCH_OK = llama_quantize_archs()
    except Exception:
        LLAMA_ARCH_OK = set(TEMPLATES)

    kept, dropped = strip_prefix(list(SRC_SHAPES))
    ARCH_SCORES = score_families(kept.values(), TEMPLATES)
    FP = fingerprint(SRC_SHAPES)

    top_arch, top_score = (ARCH_SCORES[0][0], ARCH_SCORES[0][1]) if ARCH_SCORES else (None, 0.0)
    confident = top_score >= CONFIDENT_AT

    if ARCH_MODE == "force" and ARCH_NAME.strip():
        SHIP_ARCH = QUANT_ARCH = ARCH_NAME.strip()
        VERDICT = "forced by hand in 1d"
    elif confident and top_arch in IMG_ARCH:
        SHIP_ARCH = QUANT_ARCH = top_arch
        VERDICT = "exact match, on the loader's list"
    elif confident:
        SHIP_ARCH = QUANT_ARCH = top_arch
        VERDICT = (f"exact match, but only {TEMPLATES[top_arch].get('repo', 'a fork')} "
                   f"can load it")
    elif DEFER_WHEN_UNSURE:
        SHIP_ARCH = "pig"
        QUANT_ARCH, _qwhy = pick_quant_arch(ARCH_SCORES, LLAMA_ARCH_OK, TEMPLATES, FP)
        VERDICT = "no exact match - detection deferred to load time"
    else:
        SHIP_ARCH = QUANT_ARCH = (ARCH_NAME.strip() or
            re.sub(r"[^a-z0-9]+", "_", Path(SRC_LABEL).stem.lower()).strip("_")[:24])
        VERDICT = "no match, literal name written (the loader will reject it)"

    ARCH = QUANT_ARCH                    # everything downstream converts under this
    DETECTED = [a for a, s, _ in ARCH_SCORES if s >= CONFIDENT_AT]
    DERIVED = SHIP_ARCH == "pig" or not DETECTED

    print(f"\nstructure: {FP['params'] / 1e9:.2f}B params, hidden {FP['hidden']}, "
          f"{FP['conv4d']} 4-D, {FP['nd5']} 5-D, "
          f"{'DiT' if FP['is_dit'] else 'conv unet'}")
    if FP["blocks"]:
        print("  blocks    : " + ", ".join(f"{k}={v}" for k, v in FP["blocks"].items()))
    if FP["hints"]:
        print("  looks like: " + "; ".join(FP["hints"]))
    if dropped:
        print(f"  [!] the prefix rule would drop {len(dropped)}: {dropped[:3]}")

    print("\nfamily scores:")
    for arch, score, why in ARCH_SCORES[:5]:
        mark = "  <- picked" if arch == top_arch and confident else ""
        load = "" if arch in IMG_ARCH else "  (no loader has it)"
        print(f"  {score:>5.2f}  {arch:<12} {why}{load}{mark}")

    print(f"\n{VERDICT}")
    print(f"  convert + quantize as : {QUANT_ARCH}")
    print(f"  ship as               : {SHIP_ARCH}")
    if SHIP_ARCH == "pig":
        print("  ComfyUI-GGUF skips its whitelist for 'pig' and runs its own detect_arch()")
        print("  on the tensor names at load time, so detection tracks the installed node.")
        print("  If a later release learns this family the file starts working then, with")
        print("  no reconversion. It logs a compatibility-mode warning either way.")
    elif SHIP_ARCH not in IMG_ARCH:
        print(f"  [!] {SHIP_ARCH!r} is on no loader list you pointed at. Add the fork to")
        print(f"      GGUF_NODE_REPOS, or leave DEFER_WHEN_UNSURE on to ship as 'pig'.")

# F32 protection stays structural. This is the part that decides whether a model nobody
# has written a template for works at all, and it must not wait for one - a raw
# nn.Parameter is not an nn.Linear, nothing dequantizes it at load time, and that holds
# in every architecture. A family template is a floor on top, never the whole answer.
if MODE == "diffusion" and SRC_SHAPES:
    if F32_OVERRIDE is not None:
        F32_PATTERNS = F32_OVERRIDE
        print("\nkeep in F32 (set in 1e):", ", ".join(F32_PATTERNS) or "nothing")
    else:
        from_template = list(TEMPLATES[ARCH]["keys_hiprec"]) if ARCH in TEMPLATES else []
        derived, _pats = derive_hiprec(SRC_SHAPES)
        extra = [p for p in derived
                 if p not in from_template and not any(t in p for t in from_template)]
        F32_PATTERNS = from_template + extra

        print(f"\nkeep in F32: {len(F32_PATTERNS)} pattern(s) - {len(from_template)} from "
              f"the {ARCH} template, {len(extra)} derived from these tensor names")
        if extra:
            print("  derived:", ", ".join(extra[:6]),
                  f"(+{len(extra) - 6} more)" if len(extra) > 6 else "")


### Survey a folder (optional)

Headers only, so a folder of forty checkpoints takes seconds. Needs 1a-1e and the architecture cell above to have run. For survey only, set `JOB` to `check only` in 1a so nothing tries to download.

In [ ]:
# @title 1g · Survey a folder before converting any of it { display-mode: "form" }

# @markdown Headers only, so forty checkpoints take seconds. One row each: which family
# @markdown it scores as, what it would ship as, and whether its weights are already
# @markdown quantized. Leave the path empty to skip.
SURVEY_DIR = ""  # @param {type:"string"}

if SURVEY_DIR.strip():
    folder = Path(SURVEY_DIR).expanduser().resolve()
    files = sorted(folder.rglob("*.safetensors")) if folder.is_dir() else [folder]
    print(f"{len(files)} file(s) under {folder}\n")

    head = f"{'file':<40} {'best':<12} {'conf':>5}  {'ships as':<9} note"
    print(head); print("-" * len(head))

    for f in files:
        try:
            shapes = read_header(f)
        except Exception as e:
            print(f"{f.name[:40]:<40} {'-':<12} {'-':>5}  {'-':<9} unreadable: {e}")
            continue

        kept, _ = strip_prefix(list(shapes))
        ranked = score_families(kept.values(), TEMPLATES)
        arch, conf = (ranked[0][0], ranked[0][1]) if ranked else ("none", 0.0)

        pairs, packed = find_scale_pairs(shapes)
        sub16 = [k for k, v in shapes.items()
                 if v["dtype"] in {"I8", "U8", "I4", "U4", "F8_E4M3", "F8_E5M2"}
                 and len(v["shape"]) >= 2]

        if conf >= CONFIDENT_AT and arch in IMG_ARCH:
            ships, note = arch, "clean"
        elif conf >= CONFIDENT_AT:
            ships, note = arch, "no loader on your list has this family"
        else:
            ships, note = "pig", "partial - detect_arch() decides at load"

        if packed:
            note += f"; bit-packed ({packed[0]}), cannot convert"
        elif sub16 and not pairs:
            note += f"; {len(sub16)} sub-16-bit weights, NO scales in the file"
        elif sub16:
            note += f"; {len(sub16)} sub-16-bit, {len(pairs)} scale pairs"

        print(f"{f.name[:40]:<40} {arch:<12} {conf:>5.2f}  {ships:<9} {note}")

    print("\nRows saying 'pig' still convert - they just let the node identify them.")
    print("Rows saying NO scales are the ones that come out as noise. Use the float")
    print("checkpoint for those.")
else:
    print("SURVEY_DIR empty, skipping")


## What can actually be built

An already-quantized checkpoint doesn't rule conversion out, it just limits it. This works
out what's possible before any time is spent building a toolchain, and drops the targets
that can't come out better than the source rather than failing partway through.

In [ ]:
# @title bits_of · how many bits per weight a precision label means

def bits_of(label):
    if label == "mix":
        return None
    if label.upper() in ("F16", "BF16"):
        return 16

    m = re.search(r"I?Q(\d)", label.upper())
    return int(m.group(1)) if m else None

In [ ]:
# @title apply_scale · put a weight back on its original scale

def apply_scale(weight, scale, inverse=False):
    import torch

    weight = weight.to(torch.float32)
    scale = scale.to(torch.float32)
    if inverse:
        scale = 1.0 / scale

    if scale.numel() == 1:
        return weight * scale.reshape(())
    try:
        return weight * scale                      # already broadcastable
    except RuntimeError:
        pass
    if scale.numel() == weight.shape[0]:           # one per output channel
        return weight * scale.reshape(-1, *([1] * (weight.ndim - 1)))
    if scale.numel() == weight.shape[-1]:          # one per input channel
        return weight * scale.reshape(*([1] * (weight.ndim - 1)), -1)

    raise ValueError(f"scale {tuple(scale.shape)} doesn't line up with weight "
                     f"{tuple(weight.shape)}")

In [ ]:
# @title scale_direction · multiply or divide, decided by looking at the result

def scale_direction(path, pairs, sample=8):
    """`weight_scale_inv` is named for the reciprocal but DeepSeek-style checkpoints
    dequantize with `weight * scale_inv`, not `weight / scale_inv`. Guessing from the
    name gets it backwards on half the checkpoints in the wild and the output is noise
    either way, so the direction is settled by trying both and keeping whichever lands
    the weights in a plausible range.

    Returns (inverse, note). Only the sample is read, not the model.
    """
    from safetensors import safe_open
    import torch

    if not pairs:
        return False, "no pairs"

    picked = list(pairs.items())[:sample]
    scores = {}
    with safe_open(str(path), framework="pt", device="cpu") as f:
        for inverse in (False, True):
            stds = []
            for weight, scale in picked:
                try:
                    out = apply_scale(f.get_tensor(weight), f.get_tensor(scale),
                                      inverse=inverse)
                    stds.append(float(out.to(torch.float32).std()))
                except Exception:
                    stds.append(float("nan"))
            good = sum(1 for s in stds if SANE_STD[0] <= s <= SANE_STD[1])
            scores[inverse] = (good, stds)

    fwd, inv = scores[False][0], scores[True][0]
    if fwd == inv == 0:
        med = sorted(s for s in scores[False][1] if s == s)
        return False, (f"neither direction gives sane weights (multiply -> std "
                       f"{med[len(med)//2]:.4g} on {len(med)} sampled). The scales may "
                       f"be blockwise, or this needs the original quantization library.")
    inverse = inv > fwd
    op = "weight / scale" if inverse else "weight * scale"
    return inverse, f"{op} ({max(fwd, inv)}/{len(picked)} sampled land in range)"


In [ ]:
# @title unpack_quantized · rebuild float weights from integers and scales

def unpack_quantized(state_dict):
    import torch

    pairs, packed = find_scale_pairs(state_dict.keys())
    if packed:
        raise RuntimeError(
            f"this checkpoint is bit-packed ({packed[:3]}), which needs the original "
            f"quantization library to reverse. Use the full-precision checkpoint.")

    done, failed = 0, []
    for weight, scale in pairs.items():
        try:
            state_dict[weight] = apply_scale(state_dict[weight], state_dict[scale],
                                             inverse=SCALE_INVERSE).to(torch.bfloat16)
            del state_dict[scale]
            done += 1
        except Exception as e:
            failed.append(f"{weight}: {e}")

    print(f"  rebuilt {done} weight(s) from their scales")
    if failed:
        print(f"  [!] {len(failed)} couldn't be rebuilt:")
        for f in failed[:4]:
            print("     ", f)
        raise RuntimeError("some weights couldn't be unpacked, refusing to write a broken file")

    return state_dict

In [ ]:
# @title plan_targets · work out which of the requested files are worth building

def plan_targets():
    """Nothing to weigh up if the source is 16-bit or better.

    Below that, the source only holds N bits of information per weight no matter how it gets
    stored. Writing it as BF16 doubles the file and adds nothing - which is exactly what an
    earlier version of this notebook did, turning a 12 GB int8 checkpoint into 24 GB. The
    size-matched landing spot for an 8-bit source is Q8_0, not a float type.
    """
    plan = []

    for label in EXPECT:
        want = None if label == "mix" else bits_of(label)

        if not SRC_BITS or SRC_BITS >= 16:
            plan.append((label, "build", ""))

        elif want is None:
            if BASE_QTYPE:
                plan.append((label, "build",
                             f"written as {BASE_QTYPE} - same size as the source, holds the "
                             f"unpacked values. Storing {SRC_BITS}-bit data as a float type "
                             f"would double the file for nothing"))
            else:
                plan.append((label, "too high",
                             f"a float copy of {SRC_BITS}-bit data is twice the size and no "
                             f"better"))

        elif want > SRC_BITS:
            plan.append((label, "too high",
                         f"{want}-bit from a {SRC_BITS}-bit source is about "
                         f"{want // SRC_BITS}x the size for the same quality"))

        elif want == SRC_BITS:
            plan.append((label, "build",
                         f"the size-matched target for a {SRC_BITS}-bit source"))

        else:
            plan.append((label, "build",
                         f"{want}-bit from a {SRC_BITS}-bit source stacks two rounding steps, "
                         f"but it is a real size saving"))

    return plan

In [ ]:
# @title python_quant_types · which formats gguf-py can actually produce

def python_quant_types():
    probe = np.zeros((32, 64), dtype=np.float32)
    usable = {}
    for t in gguf.GGMLQuantizationType:
        if t.name.startswith(("I", "F64", "TQ", "Q8_1", "Q8_K")):
            continue
        try:
            gguf.quants.quantize(probe, t)
            usable[t.name] = t
        except Exception:
            pass
    return usable

In [ ]:
# @title already_built · is this target already sitting in the output folder

def output_tails(label):
    """The suffix a target lands under. `mix` is whichever base type the checkpoint
    resolves to, worked out the same way convert_diffusion works it out, so which file to
    look for is known before anything has been converted."""
    if label != "mix":
        return [label.upper()]
    if BASE_QTYPE:
        return [BASE_QTYPE.upper()]
    return ["BF16", "F16"]


def already_built(label):
    """A file only counts as built when all three hold: it's named the way the converter
    would have named it, it's the full length its own header describes, and its stamp is
    the checkpoint from 1b. A name on its own is not evidence - the same filename gets
    reused across versions of a model, and reusing the wrong one is invisible until the
    output is noise."""
    if SRC is None:
        return None
    stem = SRC.name if SRC.is_dir() else SRC.stem
    tails = {t.upper() for t in output_tails(label)}

    for path in sorted(OUT.glob("*.gguf")):
        if path.stat().st_size == 0 or path.name.startswith("mmproj"):
            continue
        head, _, tail = path.stem.rpartition("-")
        if tail.upper() not in tails or head != stem:
            continue

        ok, note = gguf_is_complete(path)
        if not ok:
            print(f"  {path.name}: {note} - an interrupted write, building it again")
            continue

        verdict, detail = stamp_matches(path)
        if verdict == "different":
            print(f"  {path.name} was made from a different checkpoint ({detail}) - "
                  f"building it again")
            continue
        if verdict == "unstamped":
            print(f"  {path.name} has no source stamp, so there's no way to tell what it "
                  f"was made from - building it again")
            continue

        return path
    return None


def has_over_4d(path):
    """restore_nd writes the stashed tensors back with all their dimensions. llama-quantize
    can't read anything over 4, so a base file that has already been through that pass can
    be shipped but can't be quantized from."""
    reader = gguf.GGUFReader(str(path))
    over = any(len([d for d in t.shape if int(d)]) > 4 for t in reader.tensors)
    del reader
    return over


In [ ]:
UNPACK = False
BLOCKED = None
SCALE_INVERSE = False

if SCALES:
    print(f"this checkpoint has {len(SCALES)} scale tensor(s), so it was already quantized")
    if IF_ALREADY_QUANTIZED == "stop":
        BLOCKED = "IF_ALREADY_QUANTIZED is 'stop'"
    elif IF_ALREADY_QUANTIZED == "unpack and convert":
        pairs, packed = find_scale_pairs(SRC_SHAPES)
        if packed:
            BLOCKED = (f"bit-packed weights ({packed[:2]}) need the original quantization "
                       f"library to reverse")
        elif not pairs:
            BLOCKED = "the scale tensors don't pair up with any weight by name"
        else:
            UNPACK = True
            print(f"  {len(pairs)} weight/scale pair(s) can be rebuilt into floats")
            SCALE_INVERSE, note = scale_direction(SRC, pairs)
            print(f"  direction: {note}")
            if "neither" in note:
                BLOCKED = ("the scales don't reconstruct sane weights in either "
                           "direction - see the note above")
                UNPACK = False
    else:
        print("  [!] converting as-is: the packed integers will be written as the weights "
              "and the output will be noise")

# an 8-bit source has 8 bits of information however it is stored, so the base file is
# written at Q8_0 rather than inflated into a float type
BASE_QTYPE = None
if not BLOCKED and SRC_BITS and SRC_BITS <= 8 and MODE == "diffusion":
    if "Q8_0" in python_quant_types():
        BASE_QTYPE = "Q8_0"
        print(f"  source is {SRC_BITS}-bit, so the base file will be Q8_0 "
              f"(about the same size) instead of a float type (about double)")

if BLOCKED:
    print(f"\ncannot convert: {BLOCKED}")
    print("Use the full-precision checkpoint, or set IF_ALREADY_QUANTIZED in 1d.")
    DO_CONVERT = False
else:
    PLAN = plan_targets()
    print("\nplan:")
    for label, verdict, why in PLAN:
        mark = "build" if verdict == "build" else IF_TARGET_TOO_HIGH
        print(f"  {label:10} {mark:16} {why}")

    if any(v == "too high" for _, v, _ in PLAN) and IF_TARGET_TOO_HIGH == "stop":
        raise RuntimeError("some targets can't improve on the source and "
                           "IF_TARGET_TOO_HIGH is 'stop'")

    if IF_TARGET_TOO_HIGH == "drop it":
        keep = [l for l, v, _ in PLAN if v == "build"]
        if BASE_QTYPE and "mix" in keep and BASE_QTYPE in keep:
            keep.remove(BASE_QTYPE)          # the base file already is that type
            print(f"  {BASE_QTYPE} and mix are the same file here")
        dropped = [l for l, v, _ in PLAN if v != "build"]
        if dropped:
            print("\ndropping:", ", ".join(dropped))
        EXPECT = keep
        QUANTS = [q for q in QUANTS if q in keep]
        WANT_MIX = "mix" in keep

    if not EXPECT:
        print("\nnothing left worth building from this checkpoint.")
        print("The most useful thing you can do with it is keep it as it is.")
        DO_CONVERT = False

# anything from an earlier run that doesn't need making again. EXPECT is deliberately
# left alone so the checks still cover the kept files - they just aren't rebuilt.
REUSE_OK = IF_OUTPUT_EXISTS == "skip"
REUSE = {}
if DO_CONVERT and REUSE_OK:
    for label in EXPECT:
        found = already_built(label)
        if found:
            REUSE[label] = found

    if REUSE:
        print(f"\nalready in {OUT}:")
        for label, path in REUSE.items():
            print(f"  {label:10} {path.name}  ({path.stat().st_size / 2**30:.2f} GB)")
        QUANTS = [q for q in QUANTS if q not in REUSE]
        WANT_MIX = WANT_MIX and "mix" not in REUSE
        if all(label in REUSE for label in EXPECT):
            print("  everything asked for is there - nothing to convert")
            DO_CONVERT = False

TODO = [label for label in EXPECT if label not in REUSE]

# the base float file is always produced, quants are made from it
NEED_LLAMA = bool(QUANTS) or MODE == "llm"
print(f"\nbuilding: {', '.join(TODO) or 'nothing'}"
      + (f"  (keeping {', '.join(REUSE)})" if REUSE else ""))

# llama-quantize only accepts the architecture names its patch adds
QUANT_ENGINE = "llama"
if DO_CONVERT and MODE == "diffusion" and QUANTS:
    LLAMA_ARCH_OK = llama_quantize_archs()
    PY_TYPES = python_quant_types()

    REQUANTIZE = bool(BASE_QTYPE)
    if REQUANTIZE:
        print(f"\nthe base file is {BASE_QTYPE}, so anything further is a requantize")

    if ARCH not in LLAMA_ARCH_OK:
        print(f"\nllama-quantize doesn't know {ARCH!r}. It only accepts "
              f"{', '.join(sorted(LLAMA_ARCH_OK))}.")

        if IF_ARCH_UNKNOWN == "drop the quants":
            print("  dropping", ", ".join(QUANTS))
            EXPECT = [e for e in EXPECT if e not in QUANTS]
            QUANTS = []

        elif IF_ARCH_UNKNOWN == "stand-in arch":
            QUANT_ENGINE = "llama"
            print(f"  relabelling as {STAND_IN_ARCH!r} to quantize, then putting {ARCH!r} back")
            print(f"  [!] llama.cpp will apply {STAND_IN_ARCH!r}'s list of tensors to leave")
            print(f"      alone, and those names don't match this model. Check the dtypes in")
            print(f"      the report before shipping.")

        else:
            QUANT_ENGINE = "python"
            doable = [q for q in QUANTS if q in PY_TYPES]
            cannot = [q for q in QUANTS if q not in PY_TYPES]
            if cannot:
                print(f"  gguf-py can't build {', '.join(cannot)} - K-quants and the IQ types")
                print(f"     are only implemented in llama.cpp's C++ code. Dropping them.")
                near = {"Q4_K_M": "Q4_1", "Q4_K_S": "Q4_0", "Q5_K_M": "Q5_1",
                        "Q5_K_S": "Q5_0", "Q6_K": "Q8_0", "IQ4_NL": "Q4_1", "IQ4_XS": "Q4_0"}
                swaps = sorted({near[q] for q in cannot if q in near and near[q] in PY_TYPES})
                if swaps:
                    print(f"     closest available: {', '.join(swaps)} - tick those in 1c if "
                          f"you want them")
            if cannot and IF_QUANT_UNAVAILABLE == "substitute the nearest":
                swapped = [near[q] for q in cannot if near.get(q) in PY_TYPES]
                if swapped:
                    print(f"  substituting {', '.join(sorted(set(swapped)))}")
                    EXPECT = [near.get(e, e) if e in cannot else e for e in EXPECT]
                    doable = list(dict.fromkeys(doable + swapped))
                    cannot = [q for q in cannot if near.get(q) not in PY_TYPES]

            print(f"  quantizing in python: {', '.join(doable) or 'nothing'}")
            EXPECT = [e for e in EXPECT if e not in cannot]
            QUANTS = doable

    NEED_LLAMA = bool(QUANTS) and QUANT_ENGINE == "llama"

## Toolchain

In [ ]:
if DO_CONVERT:
    if not GG.exists():
        !git clone -q --depth 1 -b {GGUF_NODE_REF.strip() or 'main'} https://github.com/{GGUF_NODE_REPO.strip()} {str(GG)}
    print("ComfyUI-GGUF:", GG)

    if NEED_LLAMA and not (LCPP / "build" / "bin" / "llama-quantize").exists():
        if not LCPP.exists():
            !git clone -q --depth 1 -b {LLAMA_TAG.strip()} https://github.com/ggerganov/llama.cpp {str(LCPP)}
            !cd {str(LCPP)} && git apply {str(GG / 'tools' / 'lcpp.patch')} && echo "patch applied"
        print("building llama-quantize, this takes a few minutes")
        !cd {str(LCPP)} && cmake -B build -DGGML_CUDA=OFF -DLLAMA_CURL=OFF > /dev/null 2>&1
        !cd {str(LCPP)} && cmake --build build --config Release -j --target llama-quantize > /dev/null 2>&1

    QUANTIZE = LCPP / "build" / "bin" / "llama-quantize"
    print("llama-quantize:", "ready" if QUANTIZE.exists() else "not built (no quants requested)")
else:
    QUANTIZE = None
    print("check only, skipping the toolchain")

## Convert

In [ ]:
# @title convert_diffusion · one tensor at a time, so RAM never holds the whole model

def convert_diffusion(source, dst_pattern):
    """Upstream loads the entire checkpoint, then the GGUFWriter keeps every converted
    tensor as well (use_temp_file defaults to False, so add_tensor does
    self.tensors[-1][name].tensor = tensor). Peak RAM ends up above twice the model, which
    a 12 GB checkpoint can't survive on a 12.7 GB runtime.

    This reads one tensor at a time through safe_open and lets the writer spill to disk,
    so peak RAM is a few times the largest single tensor instead of the whole model.
    """
    from safetensors import safe_open
    import torch

    source = Path(source).resolve()
    files = sorted(source.rglob("*.safetensors")) if source.is_dir() else [source]

    sys.path.insert(0, str(GG / "tools"))
    import convert as base

    template = {t.arch: t for t in base.arch_list}.get(ARCH)
    shape_fix = bool(getattr(template, "shape_fix", False)) and not DERIVED
    ignore = list(getattr(template, "keys_ignore", []) or [])

    # 1. plan from the headers, without reading a single weight
    plan = []
    for path in files:
        with safe_open(str(path), framework="pt", device="cpu") as f:
            for key in f.keys():
                sl = f.get_slice(key)
                plan.append({"file": path, "key": key,
                             "dtype": sl.get_dtype(), "shape": list(sl.get_shape())})

    kept, dropped = strip_prefix([e["key"] for e in plan])
    if dropped:
        print(f"  prefix rule drops {len(dropped)} tensor(s)")

    scale_of = {}
    if UNPACK:
        pairs, _ = find_scale_pairs([e["key"] for e in plan])
        scale_of = pairs

    counts = {}
    for e in plan:
        if len(e["shape"]) >= 2 and e["key"] not in SCALES:
            counts[e["dtype"]] = counts.get(e["dtype"], 0) + 1
    main = max(counts, key=counts.get) if counts else "F16"
    base_type = None
    if BASE_QTYPE:
        # an 8-bit source stays 8-bit; writing it as a float type only doubles the file
        base_type = getattr(gguf.GGMLQuantizationType, BASE_QTYPE)
        ftype_name = BASE_QTYPE
        ftype = getattr(gguf.LlamaFileType, f"MOSTLY_{BASE_QTYPE}",
                        gguf.LlamaFileType.MOSTLY_Q8_0)
    elif UNPACK or main == "BF16":
        ftype_name, ftype = "BF16", gguf.LlamaFileType.MOSTLY_BF16
    else:
        ftype_name, ftype = "F16", gguf.LlamaFileType.MOSTLY_F16

    dst = Path(str(dst_pattern).replace("{ftype}", ftype_name)).resolve()
    dst.parent.mkdir(parents=True, exist_ok=True)

    writer = gguf.GGUFWriter(path=None, arch=ARCH, use_temp_file=True)
    writer.add_quantization_version(gguf.GGML_QUANT_VERSION)
    writer.add_file_type(ftype)
    write_stamp(writer)

    consumed = set(scale_of.values())
    todo = [e for e in plan if e["key"] not in consumed and e["key"] in kept]
    width = min(48, max((len(kept[e["key"]]) for e in todo), default=20))

    bar = None
    if PROGRESS == "summary":
        from tqdm.auto import tqdm
        bar = tqdm(total=len(todo), unit="tensor")

    held = {}
    written = 0
    tally = {}

    for path in files:
        with safe_open(str(path), framework="pt", device="cpu") as f:
            for entry in [e for e in plan if e["file"] == path]:
                key = entry["key"]
                if key in consumed or key not in kept:
                    continue
                name = kept[key]
                if any(x in name for x in ignore):
                    continue

                data = f.get_tensor(key)
                if key in scale_of:
                    data = apply_scale(data, f.get_tensor(scale_of[key]),
                                       inverse=SCALE_INVERSE)
                    data = data.to(torch.bfloat16)

                old_name = str(data.dtype).replace("torch.", "")
                old = data.dtype
                if old == torch.bfloat16:
                    arr = data.to(torch.float32).numpy()
                    qtype = gguf.GGMLQuantizationType.BF16
                elif old in (getattr(torch, "float8_e4m3fn", None),
                             getattr(torch, "float8_e5m2", None)):
                    arr = data.to(torch.float16).numpy()
                    qtype = gguf.GGMLQuantizationType.F16
                else:
                    arr = data.numpy()
                    qtype = gguf.GGMLQuantizationType.F16
                del data

                if arr.ndim > 4:
                    held[name] = torch.from_numpy(arr.copy())
                    if bar:
                        bar.update(1)
                    if PROGRESS != "quiet":
                        print(f"  {name:{width}} {old_name:>7} -> held back, "
                              f"{arr.ndim}-D {list(arr.shape)}")
                    del arr
                    continue

                n_params = int(np.prod(arr.shape))
                why = ""

                # upstream only applies these three when the source tensor is float32 or
                # bfloat16, so a weight stored as float16 - which merges and finetunes
                # leave scattered among otherwise bf16 checkpoints - skips all of them and
                # lands as F16 whatever it is. A modulation table or a norm doesn't stop
                # needing F32 because of how it happened to be stored, so the source dtype
                # source dtype isn't consulted here.
                #
                # upstream's first rule is `n_dims == 1`, which misses a raw nn.Parameter
                # stored as [1, dim] - Z-Image's cap_pad_token and x_pad_token are exactly
                # that, 3840 elements each, so they clear QUANTIZATION_THRESHOLD too and
                # land in the base file quantized. The test here is "everything but the
                # last axis is 1", which covers [dim] and [1, dim] alike.
                if int(np.prod(arr.shape[:-1])) == 1:
                    qtype, why = gguf.GGMLQuantizationType.F32, (
                        "1-D" if arr.ndim == 1 else f"single row {list(arr.shape)}")
                elif n_params <= base.QUANTIZATION_THRESHOLD:
                    qtype, why = gguf.GGMLQuantizationType.F32, "small"
                elif name in F32_PATTERNS or any(p in name for p in F32_PATTERNS):
                    qtype, why = gguf.GGMLQuantizationType.F32, "protected"
                if why and old not in (torch.float32, torch.bfloat16):
                    why += f" (was {old_name}, upstream would have left it F16)"

                if base_type is not None and qtype != gguf.GGMLQuantizationType.F32:
                    qtype, why = base_type, why or "base type"

                if (shape_fix and arr.ndim > 1 and n_params >= base.REARRANGE_THRESHOLD
                        and (n_params / 256).is_integer()
                        and not (arr.shape[-1] / 256).is_integer()):
                    writer.add_array(f"comfy.gguf.orig_shape.{name}",
                                     tuple(int(d) for d in arr.shape))
                    arr = arr.reshape(n_params // 256, 256)

                try:
                    out = gguf.quants.quantize(arr, qtype)
                except (AttributeError, gguf.QuantError) as e:
                    qtype, why = gguf.GGMLQuantizationType.F16, f"fell back: {e}"
                    out = gguf.quants.quantize(arr, qtype)

                writer.add_tensor(name, out, raw_dtype=qtype)
                written += 1
                tally[qtype.name] = tally.get(qtype.name, 0) + 1

                if PROGRESS == "every tensor":
                    print(f"  [{written:>4}/{len(todo)}] {name:{width}} {old_name:>7} -> "
                          f"{qtype.name:<5} {list(entry['shape'])}"
                          + (f"  ({why})" if why else ""))
                elif bar:
                    bar.update(1)
                    bar.set_postfix_str(name[-28:], refresh=False)
                elif why.startswith("fell back"):
                    print(f"  {name}: {why}")

                del arr, out

    if bar:
        bar.close()

    print(f"  {written} tensor(s) written: "
          + ", ".join(f"{k}={v}" for k, v in sorted(tally.items(), key=lambda x: -x[1]))
          + (f", {len(held)} held back for the restore pass" if held else ""))
    print("  writing the file out")
    writer.write_header_to_file(path=str(dst))
    writer.write_kv_data_to_file()
    writer.write_tensors_to_file(progress=True)
    writer.close()

    if held:
        from safetensors.torch import save_file
        save_file(held, str(STASH / f"fix_5d_tensors_{ARCH}.safetensors"))

    return dst, sorted(held)

In [ ]:
# @title carry_kv · move the metadata across a rewrite pass

def carry_kv(reader, writer, handled=()):
    """Every pass that rewrites a gguf - restoring the over-4D tensors, quantizing in
    python, relabelling the architecture - builds a fresh writer, and anything not copied
    over is silently gone. That matters for two keys in particular:

      comfy.gguf.orig_shape.*  is how the loader puts SD1/SDXL-style rearranged tensors
                               back. Lose it and the model loads with the wrong shapes.
      general.source.*         is the stamp saying which checkpoint this came from.

    GGUF.version, GGUF.tensor_count and GGUF.kv_count are bookkeeping the reader invents
    rather than real metadata, and writing them back produces a file that won't open.
    """
    handled = set(handled) | {"general.architecture", "general.quantization_version",
                              "general.file_type"}
    carried, lost = 0, []

    for key, field in reader.fields.items():
        if key in handled or key.startswith("GGUF.") or not field.types:
            continue
        try:
            vtype = field.types[0]
            if vtype == gguf.GGUFValueType.ARRAY:
                writer.add_array(key, [p.tolist()[0] if hasattr(p, "tolist") else p
                                       for p in (field.parts[i] for i in field.data)])
            elif vtype == gguf.GGUFValueType.STRING:
                writer.add_key_value(key, bytes(field.parts[field.data[-1]]).decode("utf-8"),
                                     vtype)
            else:
                writer.add_key_value(key, field.parts[field.data[-1]].tolist()[0], vtype)
            carried += 1
        except Exception:
            lost.append(key)

    if lost:
        print(f"    [!] couldn't carry across: {lost}")
    return carried, lost


In [ ]:
# @title restore_nd · put the stashed tensors back after quantizing

def restore_nd(path, stash_file):
    from safetensors.torch import load_file

    reader = gguf.GGUFReader(str(path))
    arch = read_field(reader, "general.architecture")
    ftype = gguf.LlamaFileType(read_field(reader, "general.file_type"))
    stash = {k: v.numpy() for k, v in load_file(str(stash_file)).items()}

    writer = gguf.GGUFWriter(path=None, arch=arch, use_temp_file=True)
    writer.add_quantization_version(gguf.GGML_QUANT_VERSION)
    writer.add_file_type(ftype)
    carry_kv(reader, writer)

    for tensor in reader.tensors:
        writer.add_tensor(tensor.name, tensor.data, raw_dtype=tensor.tensor_type)
    for key, data in stash.items():
        writer.add_tensor(key, gguf.quants.quantize(data, gguf.GGMLQuantizationType.F32),
                          raw_dtype=gguf.GGMLQuantizationType.F32)

    tmp = path.with_suffix(".fixed.gguf")
    writer.write_header_to_file(path=str(tmp))
    writer.write_kv_data_to_file()
    writer.write_tensors_to_file()
    writer.close()

    del reader
    tmp.replace(path)
    print(f"  restored {len(stash)} into {path.name}")

In [ ]:
# @title restore_from_base · undo what llama-quantize did behind the notebook's back

def restore_from_base(path, base_file):
    """llama-quantize reads the base file and writes its own opinion of it. Two of those
    opinions have to be walked back, and both are cheapest to fix in the same rewrite.

    Dimensions. llama.cpp records each tensor's dimension count as ggml_n_dims(), which
    counts trailing 1s as absent, so a [1, dim] nn.Parameter leaves as [dim] and ComfyUI
    refuses it - the module declares nn.Parameter(torch.empty((1, dim))) and load_state_dict
    raises a size mismatch, strict or not. city96's lcpp.patch adds gguf_set_tensor_ndim for
    this but only wires it to Wan's .modulation tensors by name. Only dimensions of size 1
    are put back; anything else is a real difference and not this function's to paper over.

    Protections. F32_PATTERNS is enforced by convert_diffusion and by quantize_python, and
    ignored completely by llama-quantize, which has its own per-family skip lists in
    lcpp.patch and knows nothing about this checkpoint. Anything protected here that came
    back quantized is copied out of the base file, where it is still F32.

    The base file is the reference for both: gguf-py wrote it, so it has the shapes and the
    dtypes this notebook intended.
    """
    def shape_of(tensor):
        return [int(x) for x in reversed(list(tensor.shape)) if int(x)]

    if base_file is None or not Path(base_file).exists():
        return path

    F32 = gguf.GGMLQuantizationType.F32
    protected = lambda n: n in F32_PATTERNS or any(p in n for p in F32_PATTERNS)

    base_reader = gguf.GGUFReader(str(base_file))
    base = {t.name: t for t in base_reader.tensors}
    reader = gguf.GGUFReader(str(path))

    dims, prot = {}, {}
    for tensor in reader.tensors:
        ref = base.get(tensor.name)
        if ref is None:
            continue
        if (ref.tensor_type == F32 and tensor.tensor_type != F32
                and protected(tensor.name)):
            prot[tensor.name] = ref
            continue
        want, have = shape_of(ref), shape_of(tensor)
        if want != have and [d for d in want if d != 1] == [d for d in have if d != 1]:
            dims[tensor.name] = want

    if not dims and not prot:
        del reader, base_reader
        return path

    writer = gguf.GGUFWriter(path=None, arch=read_field(reader, "general.architecture"),
                             use_temp_file=True)
    writer.add_quantization_version(gguf.GGML_QUANT_VERSION)
    writer.add_file_type(gguf.LlamaFileType(read_field(reader, "general.file_type")))
    carry_kv(reader, writer)

    for tensor in reader.tensors:
        if tensor.name in prot:
            # the base tensor is F32 and already the right shape
            writer.add_tensor(tensor.name, prot[tensor.name].data, raw_dtype=F32)
            continue
        data = tensor.data
        if tensor.name in dims:
            # blocks run along the last axis, so leading dimensions can be reshaped back
            # whether the payload is floats or packed quant blocks - add_tensor turns a
            # uint8 byte shape into the logical one from raw_dtype
            data = data.reshape(tuple(dims[tensor.name][:-1]) + (-1,))
        writer.add_tensor(tensor.name, data, raw_dtype=tensor.tensor_type)

    tmp = Path(path).with_suffix(".fixup.gguf")
    writer.write_header_to_file(path=str(tmp))
    writer.write_kv_data_to_file()
    writer.write_tensors_to_file()
    writer.close()
    del reader, base_reader
    tmp.replace(path)

    if dims:
        print(f"    put back the dimension(s) llama-quantize dropped on {len(dims)} "
              f"tensor(s): {sorted(dims)[:4]}")
    if prot:
        print(f"    llama-quantize ignored the F32 protection on {len(prot)} tensor(s), "
              f"copied back from the base: {sorted(prot)[:4]}")
    return path


In [ ]:
# @title quantize_python · quantize here when llama.cpp won't take the architecture

def quantize_python(base_file, qtype):
    """Streams the base file tensor by tensor. Uses the F32 protections worked out from this
    checkpoint rather than llama.cpp's per-family name lists, which wouldn't match anyway."""
    target = PY_TYPES[qtype]
    out = OUT / f"{base_file.name.rsplit('-', 1)[0]}-{qtype}.gguf"
    started = time.time()
    print(f"  {qtype} from {base_file.name} (in python)", flush=True)

    check_base_is_ours(base_file)

    reader = gguf.GGUFReader(str(base_file))
    arch = read_field(reader, "general.architecture")
    ftype = getattr(gguf.LlamaFileType, f"MOSTLY_{qtype}", gguf.LlamaFileType.ALL_F32)

    writer = gguf.GGUFWriter(path=None, arch=arch, use_temp_file=True)
    writer.add_quantization_version(gguf.GGML_QUANT_VERSION)
    writer.add_file_type(ftype)
    carry_kv(reader, writer)

    kept = converted = fallback = 0
    for tensor in reader.tensors:
        name = tensor.name
        shape = [int(x) for x in reversed(list(tensor.shape)) if int(x)]
        n_params = int(np.prod(shape)) if shape else 0

        # len(shape) < 2 would let [1, dim] through, and that is one row - as unsafe to
        # quantize as a 1-D tensor, for the same reason
        protect = (int(np.prod(shape[:-1] or [1])) == 1 or len(shape) > 4
                   or n_params <= 1024
                   or name in F32_PATTERNS
                   or any(p in name for p in F32_PATTERNS))

        if protect:
            writer.add_tensor(name, tensor.data, raw_dtype=tensor.tensor_type)
            kept += 1
            continue

        try:
            arr = gguf.quants.dequantize(tensor.data, tensor.tensor_type).astype(np.float32)
            arr = arr.reshape(shape)
            writer.add_tensor(name, gguf.quants.quantize(arr, target), raw_dtype=target)
            converted += 1
        except Exception:
            writer.add_tensor(name, tensor.data, raw_dtype=tensor.tensor_type)
            fallback += 1
        finally:
            arr = None

        if PROGRESS == "every tensor":
            print(f"    {name} -> {target.name}")

    writer.write_header_to_file(path=str(out))
    writer.write_kv_data_to_file()
    writer.write_tensors_to_file(progress=(PROGRESS != "quiet"))
    writer.close()
    del reader

    before = base_file.stat().st_size / 2**30
    after = out.stat().st_size / 2**30
    print(f"    {converted} quantized, {kept} left alone"
          + (f", {fallback} couldn't be converted" if fallback else ""))
    print(f"    {after:.2f} GB, {after / before:.0%} of the float file, "
          f"{time.time() - started:.0f}s")
    return out

In [ ]:
# @title relabel_arch · change the architecture string, keep everything else

def relabel_arch(path, arch):
    """Only the metadata string changes - the tensors are copied across untouched, so this
    is not a reconversion. It does write a second copy first, so you need as much free disk
    as the file is big."""
    reader = gguf.GGUFReader(str(path))
    ftype = gguf.LlamaFileType(read_field(reader, "general.file_type"))

    writer = gguf.GGUFWriter(path=None, arch=arch, use_temp_file=True)
    writer.add_quantization_version(gguf.GGML_QUANT_VERSION)
    writer.add_file_type(ftype)

    # comfy.gguf.orig_shape.* and the source stamp both live in here
    carried, lost = carry_kv(reader, writer)

    for tensor in reader.tensors:
        writer.add_tensor(tensor.name, tensor.data, raw_dtype=tensor.tensor_type)

    tmp = path.with_suffix(".relabel.gguf")
    writer.write_header_to_file(path=str(tmp))
    writer.write_kv_data_to_file()
    writer.write_tensors_to_file()
    writer.close()
    del reader

    tmp.replace(path)
    return path, carried

In [ ]:
# @title check_base_is_ours · refuse to cut a quant from the wrong file

def check_base_is_ours(base_file):
    """Every quant inherits whatever the base file holds, so this is the last point where
    a mix-up is still cheap to catch. A base from another checkpoint, or one left half
    written by a run that was interrupted, produces a quant that loads and generates
    noise - which looks like a quantization problem and isn't."""
    ok, note = gguf_is_complete(base_file)
    if not ok:
        raise RuntimeError(
            f"{base_file.name}: {note}. Delete it and convert again rather than "
            f"quantizing a file that was left half written.")

    verdict, detail = stamp_matches(base_file)
    if verdict == "different":
        raise RuntimeError(
            f"{base_file.name} was not made from the checkpoint in 1b - {detail}. "
            f"Delete it, or point 1b at the checkpoint it really came from.")
    if verdict == "unstamped":
        print(f"  [!] {base_file.name} carries no source stamp, so there's no way to "
              f"confirm it came from this checkpoint")
    elif verdict == "match":
        print(f"  base file checks out: {detail}")


In [ ]:
# @title quantize · one llama-quantize pass

def quantize(base_file, qtype):
    if QUANT_ENGINE == "python":
        return quantize_python(base_file, qtype)

    check_base_is_ours(base_file)
    out = OUT / f"{base_file.name.rsplit('-', 1)[0]}-{qtype}.gguf"
    started = time.time()
    print(f"  {qtype} from {base_file.name}", flush=True)

    # llama-quantize refuses a quantized input unless told otherwise:
    # "requantizing from type q8_0 is disabled"
    cmd = [str(QUANTIZE)]
    if REQUANTIZE:
        cmd.append("--allow-requantize")
    cmd += [str(base_file), str(out), qtype]

    proc = subprocess.Popen(cmd,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1)
    tail = []
    for line in proc.stdout:
        tail.append(line)
        if PROGRESS == "every tensor":
            print("   ", line.rstrip())
        elif "converting to" in line or "quantize time" in line:
            print("   ", line.rstrip())
    if proc.wait() != 0:
        print("".join(tail[-25:]))
        raise RuntimeError(f"llama-quantize failed for {qtype}")

    # llama.cpp writes ggml_n_dims(), which ignores trailing 1s, and enforces its own
    # per-family F32 list rather than this checkpoint's - both are compared against the
    # base file and put back
    restore_from_base(out, base_file)

    if SRC_FP and stamp_matches(out)[0] != "match":
        # llama.cpp copies the metadata forward, so this only fires if that changed
        print("    [!] the source stamp didn't survive llama-quantize, putting it back")
        relabel_arch(out, read_field(gguf.GGUFReader(str(out)), "general.architecture"))

    before = base_file.stat().st_size / 2**30
    after = out.stat().st_size / 2**30
    print(f"    {after:.2f} GB, {after / before:.0%} of the float file, "
          f"{time.time() - started:.0f}s")
    return out

In [ ]:
BUILT = []

if DO_CONVERT and MODE == "diffusion":
    for stale in STASH.glob("fix_5d_tensors_*.safetensors"):
        stale.unlink()
        print("cleared a stale stash file from an earlier run:", stale.name)

    name = SRC.stem
    started = time.time()

    # every quant is cut from the base file, so one left over from an earlier run is worth
    # reusing even when 1c didn't ask for the base itself. The exception is a file that has
    # already been through restore_nd - llama-quantize can't read the tensors it put back,
    # so that one has to be converted again before anything can be quantized from it.
    base_file, nd = (REUSE.get("mix") or (already_built("mix") if REUSE_OK else None)), []
    if base_file is not None and QUANTS and has_over_4d(base_file):
        print(f"{base_file.name} already has its over-4D tensors back in it, which "
              f"llama-quantize can't read - converting again")
        base_file = None

    if base_file is None:
        print(f"converting {SRC.name} ({SRC.stat().st_size / 2**30:.2f} GB) as {ARCH!r}")
        base_file, nd = convert_diffusion(SRC, str(OUT / f"{name}-{{ftype}}.gguf"))
        BUILT.append(base_file)
        print("wrote", base_file.name)
    else:
        print(f"reusing {base_file.name} as the base, not converting again")

    if QUANTS:
        print(f"\nquantizing to {', '.join(QUANTS)}")

    stand_in = (IF_ARCH_UNKNOWN == "stand-in arch" and QUANTS
                and ARCH not in llama_quantize_archs())
    if stand_in:
        relabel_arch(base_file, QUANT_ARCH)

    for qtype in QUANTS:
        BUILT.append(quantize(base_file, qtype))

    if stand_in:
        touched = list(dict.fromkeys(BUILT + [base_file]))
        for f in touched:
            relabel_arch(f, ARCH)
        print(f"  put {ARCH!r} back on {len(touched)} file(s)")

    # last step before the checks: the name the file ships under. Everything up to here
    # ran as QUANT_ARCH because llama-quantize only accepts names lcpp.patch adds; only
    # the metadata string changes, the tensors are copied across untouched.
    if SHIP_ARCH and SHIP_ARCH != ARCH:
        for f in list(dict.fromkeys(BUILT + [base_file])):
            if f.exists():
                relabel_arch(f, SHIP_ARCH)
        print(f"\nshipping as {SHIP_ARCH!r} (converted as {ARCH!r})")

    if nd:
        stash_file = STASH / f"fix_5d_tensors_{ARCH}.safetensors"
        print(f"\nrestoring {len(nd)} tensor(s) over 4 dimensions")
        for f in BUILT:
            restore_nd(f, stash_file)
        stash_file.unlink()
    print(f"\nconversion took {time.time() - started:.0f}s")

    # WANT_MIX goes false when the base was reused, so ask EXPECT instead - otherwise a
    # kept file gets deleted as if it were this run's scratch
    if "mix" not in EXPECT and base_file in BUILT:
        base_file.unlink()
        BUILT.remove(base_file)
        print("dropped the intermediate float file")

elif DO_CONVERT and MODE == "llm":
    conv = LCPP / "convert_hf_to_gguf.py"
    outtype = "auto" if WANT_MIX else "f16"

    reused = REUSE.get("mix") or (already_built("mix") if REUSE_OK else None)
    if reused is not None:
        print(f"reusing {reused.name} as the base, not converting again")
    else:
        cmd = ["python", str(conv), str(SRC), "--outfile",
               str(OUT / f"{SRC.name}-{{ftype}}.gguf"), "--outtype", outtype]
        print(" ".join(cmd))
        subprocess.run(cmd, check=True)
    BUILT = sorted(OUT.glob("*.gguf"))

    if WANT_MMPROJ and REUSE_OK and any(p.name.startswith("mmproj") for p in BUILT):
        print("mmproj is already there, leaving it")
    elif WANT_MMPROJ:
        r = subprocess.run(["python", str(conv), str(SRC), "--mmproj", "--outfile",
                            str(OUT / f"mmproj-{SRC.name}.gguf")],
                           capture_output=True, text=True)
        if r.returncode != 0:
            print("mmproj export failed:\n", (r.stdout + r.stderr)[-2000:])
        else:
            BUILT = sorted(OUT.glob("*.gguf"))

    base_file = max([p for p in BUILT if not p.name.startswith("mmproj")],
                    key=lambda p: p.stat().st_size, default=None)
    for qtype in QUANTS:
        BUILT.append(quantize(base_file, qtype))

else:
    print("check only, nothing converted")

print("\nbuilt")
for p in sorted(OUT.glob("*.gguf")):
    label = p.stem.rsplit("-", 1)[-1] if "-" in p.stem else ""
    print(f"  {p.name:52} {p.stat().st_size / 2**30:>6.2f} GB  {label}")

others = [p for p in sorted(OUT.glob("*")) if p.is_file() and p.suffix != ".gguf"]
if others:
    print("alongside")
    for p in others:
        print(f"  {p.name:52} {p.stat().st_size / 2**30:>6.2f} GB")

## Check what came out

In [ ]:
QUANT_TYPES = {t for t in gguf.GGMLQuantizationType
               if t.name not in ("F32", "F16", "BF16", "F64", "I8", "I16", "I32", "I64")}

FILES = sorted(OUT.rglob("*.gguf"))
MMPROJ = [p for p in FILES if p.name.startswith("mmproj")]
MODELS = [p for p in FILES if p not in MMPROJ]
BASE = max(MODELS, key=lambda p: p.stat().st_size) if MODELS else None

REPORT = {}
_current = None
_kind = "model"

if not FILES:
    print(f"no .gguf under {OUT}")
    loose = [p for p in OUT.rglob("*") if p.is_file()]
    for p in loose[:10]:
        print("  ", p.relative_to(OUT))
print("checking:", ", ".join(p.name for p in FILES) or "nothing")

In [ ]:
# @title log · record one result and print it

def log(level, msg):
    REPORT.setdefault(_current, []).append((level, msg))
    print(f"  [{level}] {msg}")

In [ ]:
# @title read_gguf · tensor names, shapes and types

def read_gguf(path):
    reader = gguf.GGUFReader(str(path))
    tensors = {}
    for t in reader.tensors:
        # gguf stores dimensions in the opposite order to torch
        shape = [int(x) for x in reversed(list(t.shape)) if int(x) != 0]
        tensors[t.name] = {"shape": shape, "type": t.tensor_type, "tensor": t}
    return reader, tensors

In [ ]:
# @title check_files_exist · did everything from 1c actually get made

def check_files_exist():
    names = [p.name.upper() for p in FILES]

    if not FILES:
        log("FAIL", "nothing to check - 1c asked for " + ", ".join(EXPECT))
        return

    for label in EXPECT:
        if label == "mix":
            want_end = ([f"-{BASE_QTYPE.upper()}.GGUF"] if BASE_QTYPE
                        else ["-BF16.GGUF", "-F16.GGUF"])
            found = [n for n in names if any(n.endswith(w) for w in want_end)]
            what = f"mix ({BASE_QTYPE})" if BASE_QTYPE else "mix (BF16 or F16)"
        else:
            found = [n for n in names if n.endswith(f"-{label.upper()}.GGUF")]
            what = label
        log("PASS" if found else "FAIL",
            f"{what}: {found[0].lower()}" if found else f"{what}: no file")

    if WANT_MMPROJ and not MMPROJ:
        log("FAIL", "no mmproj file, so the image or audio side is missing and the model "
                    "will only do text")
    elif WANT_MMPROJ:
        log("PASS", f"mmproj: {MMPROJ[0].name}")

    for what, patterns in COMPANIONS:
        hits = [p.name for pat in patterns for p in OUT.rglob(pat) if p.is_file()]
        log("PASS" if hits else "FAIL",
            f"{what}: {', '.join(sorted(set(hits))[:3])}" if hits else f"{what}: missing")

    if MODE == "diffusion" and not COMPANIONS:
        log("WARN", "no companion files ticked in 1e. A diffusion gguf is only the "
                    "transformer, the VAE and text encoder have to ship with it")

In [ ]:
# @title check_arch · will the runtime accept this architecture string

def check_arch(reader):
    arch = read_field(reader, "general.architecture")
    if not arch:
        log("FAIL", "no general.architecture in the file")
        return

    # "pig" and "cow" make the loader skip its whitelist and detect from the tensor
    # names instead, so they are valid on purpose, not a fallback that slipped through
    if arch in ("pig", "cow"):
        log("PASS", f"arch is {arch!r} - the loader will detect from the tensor names "
                    f"at load time, using whatever ComfyUI-GGUF is installed")
        if DETECTED:
            log("INFO", f"the weights score highest as {DETECTED[0]!r}, which is what "
                        f"detect_arch() should land on")
        else:
            log("WARN", "no family here scored a full match, so whether this loads "
                        "depends on the installed node knowing the architecture")
        return

    if TARGET.startswith("ComfyUI"):
        allowed = TXT_ARCH if _kind == "textenc" else IMG_ARCH
        if arch in allowed:
            log("PASS", f"arch is {arch!r}, ComfyUI will accept it")
        else:
            log("FAIL", f"arch is {arch!r}, not in ComfyUI's list, the loader will reject "
                        f"it until the node supports this architecture")
    else:
        log("INFO", f"arch is {arch!r}")

    if MODE == "diffusion" and DETECTED and arch != DETECTED[0]:
        if arch in DETECTED:
            log("WARN", f"the weights match {DETECTED}, those families share detection keys")
        else:
            log("FAIL", f"the file says {arch!r} but the weights match {DETECTED[0]!r}")
    elif MODE == "diffusion" and DERIVED:
        log("INFO", f"{arch!r} was derived, no upstream family matched these weights")

In [ ]:
# @title check_source · same tensors, same shapes as the checkpoint

def check_source(gg):
    # unpacking folds each scale into its weight, so the scales are meant to be gone
    consumed = set(SCALES) if UNPACK else set()
    kept, dropped = strip_prefix([k for k in SRC_SHAPES if k not in consumed])
    if dropped:
        log("FAIL", f"prefix stripping threw away {len(dropped)} source tensors: {dropped[:4]}")

    want = {new: SRC_SHAPES[old]["shape"] for old, new in kept.items()}
    missing = sorted(set(want) - set(gg))
    extra = sorted(set(gg) - set(want))

    big = [k for k in missing if len(want[k]) > 4]
    rest = [k for k in missing if len(want[k]) <= 4]

    if big:
        log("FAIL", f"{len(big)} tensor(s) over 4 dimensions never made it back in: {big[:4]}")
    if rest:
        log("FAIL", f"{len(rest)} source tensor(s) missing: {rest[:8]}")
    if extra:
        log("WARN", f"{len(extra)} tensor(s) with no source counterpart: {extra[:8]}")
    if not missing and not extra:
        log("PASS", f"same tensors as the source ({len(gg)})"
                    + (f", {len(consumed)} scale tensor(s) folded into their weights"
                       if consumed else ""))

    bad = [(k, want[k], gg[k]["shape"]) for k in sorted(set(want) & set(gg))
           if list(want[k]) != list(gg[k]["shape"])]
    if bad:
        log("FAIL", f"{len(bad)} shape mismatch(es):")
        for k, a, b in bad[:8]:
            print(f"           {k}: source {a} -> gguf {b}")
    else:
        log("PASS", "shapes all match")

In [ ]:
# @title check_dtypes · nothing quantized that should have stayed F32

def check_dtypes(gg):
    dist = {}
    for m in gg.values():
        dist[m["type"].name] = dist.get(m["type"].name, 0) + 1
    log("INFO", "dtypes: " + ", ".join(f"{k}={v}" for k, v in
                                       sorted(dist.items(), key=lambda x: -x[1])))

    # a [1, dim] parameter is a single row, exactly as unsafe to quantize as a 1-D
    # tensor. Testing len(shape) == 1 passes it, which is how cap_pad_token and
    # x_pad_token went out as Q8_0 in a file this check called clean - the shape only
    # collapses to [dim] once llama-quantize has rewritten it, so the base file looked
    # fine and the quant cut from it did not
    flat = [n for n, m in gg.items()
            if int(np.prod(m["shape"][:-1] or [1])) == 1 and m["type"] in QUANT_TYPES]
    if flat:
        log("FAIL", f"{len(flat)} single-row tensor(s) quantized - norms, biases and raw "
                    f"nn.Parameters need F32: {flat[:6]}")
    else:
        log("PASS", "no single-row tensor was quantized")

    missed = [p for p in F32_PATTERNS
              if p in gg and gg[p]["type"] != gguf.GGMLQuantizationType.F32]
    loose = [n for n, m in gg.items()
             if m["type"] != gguf.GGMLQuantizationType.F32
             and any(p in n for p in F32_PATTERNS if p not in gg)]
    wrong = sorted(set(missed) | set(loose))

    # a pattern that matches no tensor at all protects nothing, and without this the
    # check passes for having found nothing to look at - which is how a prefixed or
    # misspelled entry hides a weight that was never protected
    dead = [p for p in F32_PATTERNS if p not in gg and not any(p in n for n in gg)]

    if not F32_PATTERNS:
        log("INFO", "this family has no weights needing F32 protection")
    elif wrong:
        log("FAIL", f"{len(wrong)} tensor(s) that must stay F32 were quantized: {wrong[:5]}")
    elif dead and len(dead) == len(F32_PATTERNS):
        log("FAIL", f"none of the {len(dead)} protect pattern(s) match any tensor in this "
                    f"file, so nothing was protected: {dead[:5]}")
    elif dead:
        log("WARN", f"{len(dead)} of {len(F32_PATTERNS)} protect pattern(s) match no tensor "
                    f"here: {dead[:5]}")
        log("PASS", f"the other {len(F32_PATTERNS) - len(dead)} protected tensor(s) are F32")
    else:
        log("PASS", f"all {len(F32_PATTERNS)} protected tensor(s) are F32")

In [ ]:
# @title check_tokenizer · vocab, embeddings and chat template line up

def check_tokenizer(reader, gg):
    for key in ("tokenizer.ggml.tokens", "tokenizer.ggml.token_type"):
        if reader.fields.get(key) is None:
            log("FAIL", f"{key} missing, the model won't tokenize")
        else:
            log("PASS", f"{key} present")

    n_vocab = None
    f = reader.fields.get("tokenizer.ggml.tokens")
    if f is not None:
        n_vocab = len(f.data)
        log("INFO", f"vocab is {n_vocab}")

    for name in ("token_embd.weight", "output.weight"):
        if name in gg and n_vocab:
            rows = gg[name]["shape"][0]
            if rows != n_vocab:
                log("FAIL", f"{name} has {rows} rows, tokenizer says {n_vocab}")
            else:
                log("PASS", f"{name} matches the vocab ({rows})")

    if SUBTASK in ("text", "vision", "audio") and reader.fields.get("tokenizer.chat_template") is None:
        log("WARN", "no chat template, the runtime will guess a format")

In [ ]:
# @title check_mmproj · is the image or audio tower actually in there

def check_mmproj(gg):
    vision = any(n.startswith("v.") for n in gg)
    audio = any(n.startswith("a.") for n in gg)
    log("INFO", f"vision={vision} audio={audio}")

    if not (vision or audio):
        log("FAIL", "no v.* or a.* tensors, this projector is empty")
    if SUBTASK == "audio" and not audio:
        log("FAIL", "audio task but no a.* tensors, the audio export failed quietly "
                    "and the model is text-only")
    if SUBTASK == "vision" and not vision:
        log("FAIL", "vision task but no v.* tensors")

In [ ]:
# @title check_vs_base · the quant kept everything the float file had

def check_vs_base(gg, base):
    missing = sorted(set(base) - set(gg))
    extra = sorted(set(gg) - set(base))
    bad = [n for n in sorted(set(gg) & set(base))
           if list(gg[n]["shape"]) != list(base[n]["shape"])]

    if missing:
        log("FAIL", f"{len(missing)} tensor(s) lost in quantization: {missing[:8]}")
    if extra:
        log("WARN", f"{len(extra)} tensor(s) the base doesn't have: {extra[:8]}")
    if bad:
        log("FAIL", f"{len(bad)} tensor(s) changed shape during quantization: {bad[:8]}")
    if not (missing or extra or bad):
        log("PASS", f"same tensors and shapes as {BASE.name}")

In [ ]:
# @title check_nan · dequantize a sample and look for broken numbers

def check_nan(gg):
    names = list(gg)
    if NAN_SAMPLE and len(names) > NAN_SAMPLE:
        names = names[::max(1, len(names) // NAN_SAMPLE)]

    checked = skipped = 0
    bad = []
    for n in names:
        t = gg[n]["tensor"]
        try:
            if t.tensor_type in (gguf.GGMLQuantizationType.F32, gguf.GGMLQuantizationType.F16):
                arr = t.data.astype(np.float32)
            else:
                arr = gguf.quants.dequantize(t.data, t.tensor_type).astype(np.float32)
        except Exception:
            skipped += 1
            continue
        checked += 1
        if not np.isfinite(arr).all():
            bad.append(n)

    for n in bad:
        log("FAIL", f"NaN or Inf in {n!r}")
    if not bad and checked:
        log("PASS", f"no NaN in {checked} sampled tensors ({skipped} couldn't be read)")

In [ ]:
# @title check_stamp · is this file complete, and from the checkpoint in 1b

def check_stamp(path):
    ok, note = gguf_is_complete(path)
    log("PASS" if ok else "FAIL", f"file is complete: {note}" if ok
        else f"file was left half written: {note}")

    verdict, detail = stamp_matches(path)
    if verdict == "match":
        log("PASS", f"made from {detail}")
    elif verdict == "different":
        log("FAIL", f"not made from the checkpoint in 1b - {detail}")
    elif verdict == "unstamped":
        log("WARN", detail)
    else:
        log("INFO", detail)


In [ ]:
# @title check_magnitude · are these numbers the size a weight should be

def check_magnitude(gg, path, sample=12):
    """The check that catches a dropped scale. Nothing about names or shapes is wrong
    when the packed integers get written out as the weights - every tensor is present,
    every shape matches, every dtype is legal. Only the values give it away: a real
    diffusion weight has a standard deviation around 0.005 to 0.1, and raw int8 codes
    land around 10 to 80. Both files pass every other check in this notebook.
    """
    reader = gguf.GGUFReader(str(path))
    big = sorted((t for t in reader.tensors
                  if t.tensor_type not in (gguf.GGMLQuantizationType.F32,)
                  and len([d for d in t.shape if d > 1]) >= 2),
                 key=lambda t: -int(np.prod([int(d) for d in t.shape if d])))[:sample]

    if not big:
        log("INFO", "no multi-row quantized tensors to measure")
        del reader
        return

    stds, bad = [], []
    for t in big:
        try:
            arr = gguf.quants.dequantize(t.data, t.tensor_type).astype(np.float32)
        except Exception:
            continue
        s = float(arr.reshape(-1).std())
        stds.append((t.name, s))
        if not (SANE_STD[0] <= s <= SANE_STD[1]):
            bad.append((t.name, s))
        del arr

    del reader
    if not stds:
        log("INFO", "nothing could be dequantized to measure")
        return

    med = sorted(s for _, s in stds)[len(stds) // 2]

    # what counts as a normal weight is a property of the model, not a constant. A merge
    # can sit well above the usual 0.02 and be entirely correct. When the checkpoint is
    # on hand it settles the question outright, so only fall back to the fixed band when
    # there is nothing to compare against.
    src_med = None
    if SRC is not None and SRC_SHAPES:
        try:
            from safetensors import safe_open
            import torch
            kept, _ = strip_prefix(list(SRC_SHAPES))
            tail = {t: o for o, t in kept.items()}
            picks = [tail[n] for n, _ in stds if n in tail][:6]
            if picks:
                vals = []
                with safe_open(str(SRC), framework="pt", device="cpu") as f:
                    for k in picks:
                        t = f.get_tensor(k).to(torch.float32)
                        if k in SRC_PAIRS:
                            t = apply_scale(t, f.get_tensor(SRC_PAIRS[k]),
                                            inverse=SCALE_INVERSE).to(torch.float32)
                        vals.append(float(t.std()))
                src_med = sorted(vals)[len(vals) // 2]
        except Exception:
            src_med = None

    if src_med:
        ratio = med / (src_med or 1e-12)
        if 0.5 <= ratio <= 2.0:
            log("PASS", f"weight magnitudes match the checkpoint (median std {med:.4g} "
                        f"here, {src_med:.4g} in the source)")
        else:
            log("FAIL", f"weight magnitudes are {ratio:.2f}x the checkpoint's (median "
                        f"std {med:.4g} here, {src_med:.4g} in the source). A scale "
                        f"applied in the wrong direction, or applied twice, lands here.")
        return

    # nothing legitimate in a diffusion model has a standard deviation in the tens.
    # A value that size is an int8 code that never got multiplied by its scale, and it
    # is a hard failure however few tensors show it - the file cannot produce an image.
    insane = [(n, s) for n, s in stds if s > 5 or s != s]
    if insane:
        log("FAIL", f"{len(insane)}/{len(stds)} sampled weights have std "
                    f"{insane[0][1]:.3g} and up, e.g. {insane[0][0]}. Values this size "
                    f"are packed integer codes, not weights - the quantization scales "
                    f"were never applied and this file will render as noise.")
    elif len(bad) > len(stds) // 2:
        log("FAIL", f"{len(bad)}/{len(stds)} sampled weights sit outside "
                    f"{SANE_STD[0]}..{SANE_STD[1]} (median {med:.4g}): "
                    f"{[n for n, _ in bad[:3]]}. Something rescaled these weights.")
    elif bad:
        log("WARN", f"{len(bad)}/{len(stds)} sampled weights look unusual "
                    f"(median {med:.4g}): {[n for n, _ in bad[:3]]}")
    elif med > 0.08:
        log("WARN", f"no checkpoint to compare against, and the median std is {med:.4g} "
                    f"across {len(stds)} sampled where a diffusion weight is usually "
                    f"nearer 0.02. Often just this model, but a scale applied in the "
                    f"wrong direction lands here too.")
    else:
        log("PASS", f"weight magnitudes are plausible (median std {med:.4g} across "
                    f"{len(stds)} sampled)")


In [ ]:
# @title check_ops_coverage · weights nothing will dequantize at runtime

def role_of(name):
    for role, pat in ROLES:
        if re.search(pat, name, re.I):
            return role
    return "other"


def check_ops_coverage(gg):
    """ComfyUI-GGUF replaces five layer types and no others: Linear, Conv2d, Embedding,
    LayerNorm, GroupNorm. Those know how to dequantize a GGMLTensor on the way through.
    A weight consumed by anything else - an RMSNorm, a raw nn.Parameter, a custom module -
    is read straight off the tensor as numbers, still packed.

    Nothing errors when this happens. The file loads, every name and shape is right, and
    the packed bytes are simply used as the weight. The output keeps local detail and
    loses everything that depends on those weights.

    A Linear or Conv weight is 2-D or 4-D, named .weight, and has more than one row.
    Anything else that is quantized is on its own.
    """
    raw = []
    for name, m in gg.items():
        if m["type"] not in QUANT_TYPES:
            continue
        shape = m["shape"]
        linear_like = (len(shape) in (2, 4) and name.endswith(".weight")
                       and int(np.prod(shape[:-1] or [1])) > 1)
        if not linear_like:
            raw.append((name, m["type"].name, shape))

    if not raw:
        log("PASS", "every quantized tensor is a Linear or Conv weight, so a GGMLOps "
                    "layer will dequantize it")
        return

    by_role = {}
    for name, ttype, shape in raw:
        by_role.setdefault(role_of(name), []).append(name)
    summary = ", ".join(f"{k} ({len(v)})" for k, v in
                        sorted(by_role.items(), key=lambda kv: -len(kv[1])))
    log("FAIL", f"{len(raw)} quantized tensor(s) that no GGMLOps layer will dequantize - "
                f"they will be used as packed bytes: {summary}")
    for name, ttype, shape in raw[:6]:
        log("INFO", f"  {name}  {ttype}  {shape}")


In [ ]:
# @title check_values · did the numbers survive, not just the names

def check_values(gg, path):
    """Names and shapes matching proves nothing. A weight that was rescaled, transposed,
    or written before its quantization scales were applied has exactly the right name and
    exactly the right shape.

    Rebuilds each source tensor - applying scales when the checkpoint is packed - and
    compares it to the same tensor dequantized back out of the gguf. What to expect when
    conversion was clean:

        F16 / BF16   below 0.001      Q4_K_M   0.02 to 0.08
        Q8_0         0.002 to 0.006   Q4_0     0.04 to 0.12

    Read the per-role table, not the tensor list. One role an order of magnitude above
    the others names the stage of the forward pass that broke.
    """
    from safetensors import safe_open
    import torch

    kept, _ = strip_prefix(list(SRC_SHAPES))
    by_tail = {tail: orig for orig, tail in kept.items()
               if orig not in set(SRC_PAIRS.values())}

    # largest few per role - the big weights carry the model and are the ones a
    # conversion fault shows up in first
    picked, seen = [], {}
    for name in gg:
        if name not in by_tail:
            continue
        src_key = by_tail[name]
        n = int(np.prod(SRC_SHAPES[src_key]["shape"]) or 0)
        seen.setdefault(role_of(name), []).append((n, name, src_key))
    for role, items in seen.items():
        items.sort(reverse=True)
        picked += [(role, nm, sk) for _, nm, sk in items[:DIFF_SAMPLE]]

    if not picked:
        log("INFO", "no tensor in this file matches the checkpoint by name")
        return

    per, worst, errors = {}, [], {}
    with safe_open(str(SRC), framework="pt", device="cpu") as f:
        for role, name, src_key in picked:
            try:
                src = f.get_tensor(src_key).to(torch.float32)
                if src_key in SRC_PAIRS:
                    src = apply_scale(src, f.get_tensor(SRC_PAIRS[src_key]),
                                      inverse=SCALE_INVERSE).to(torch.float32)
                src = src.numpy().reshape(-1)
                out = gguf.quants.dequantize(gg[name]["tensor"].data,
                                             gg[name]["type"]).astype(np.float32).reshape(-1)
            except Exception as e:
                errors.setdefault(f"{type(e).__name__}: {e}", []).append(name)
                continue

            if out.size < src.size:
                log("FAIL", f"{name} has {out.size} values in the gguf and {src.size} in "
                            f"the checkpoint")
                continue
            out = out[:src.size]

            denom = float(np.sqrt((src ** 2).mean())) or 1e-12
            rel = float(np.sqrt(((src - out) ** 2).mean())) / denom
            norm = float(np.linalg.norm(src) * np.linalg.norm(out)) or 1e-12
            cos = float(src @ out / norm)
            per.setdefault(role, []).append((rel, cos, gg[name]["type"].name))
            worst.append((rel, cos, name, gg[name]["type"].name))
            del src, out

    for msg, names in errors.items():
        log("WARN", f"{len(names)} tensor(s) couldn't be compared - {msg} "
                    f"(e.g. {names[0]})")

    if not per:
        log("WARN", "nothing could be compared, so nothing here says whether the values "
                    "in this file match the checkpoint")
        return

    rows = [(r, c, t) for v in per.values() for r, c, t in v]
    med_all = float(np.median([r for r, _, _ in rows]))

    # Q4_K_M is a mixture - llama-quantize leaves some tensors at Q6_K or Q8_0 and cuts
    # the rest to Q4_K. Comparing every role against one file-wide median makes ordinary
    # Q4_K rounding look like a fault next to a tensor that stayed at Q8_0. A tensor is
    # only unusual next to other tensors carrying the same number of bits.
    by_type = {}
    for r, _, t in rows:
        by_type.setdefault(t, []).append(r)
    # the baseline for a type comes from its lower quartile, not its median. If half the
    # tensors of a type are the broken ones they pull the median up with them and hide
    # behind it - the cleanest quarter is what ordinary rounding at that bit depth costs.
    med_type = {t: float(np.percentile(v, 25)) or 1e-9 for t, v in by_type.items()}

    log("INFO", f"value check: {len(rows)} tensor(s), median relative error "
                f"{med_all:.5f}")
    if len(med_type) > 1:
        log("INFO", "  by type: " + ", ".join(
            f"{t} {med_type[t]:.4f} (n={len(by_type[t])})"
            for t in sorted(med_type, key=lambda k: med_type[k])))

    off = []
    def norm(v):
        return float(np.median([r / med_type[t] for r, _, t in v]))

    for role in sorted(per, key=lambda k: -norm(per[k])):
        v = per[role]
        med = float(np.median([r for r, _, _ in v]))
        ratio = norm(v)
        types = "/".join(sorted({t for _, _, t in v}))
        bad = ratio > 4.0 and med > 0.02
        if bad:
            off.append(role)
        log("INFO", f"  {role:<12} n={len(v):<3} {types:<14} median {med:>9.5f}  "
                    f"{ratio:>5.1f}x its type" + ("   <-- out of line" if bad else ""))

    worst.sort(reverse=True)
    if off:
        log("FAIL", f"{', '.join(off)} {'are' if len(off) > 1 else 'is'} far off other "
                    f"tensors carrying the same number of bits, so this is not "
                    f"quantization loss - those weights changed. Worst: "
                    f"{worst[0][2]} ({worst[0][3]}, rel {worst[0][0]:.4f}, "
                    f"cos {worst[0][1]:.4f})")
    elif med_all > 0.15:
        log("FAIL", f"every role is off by {med_all:.3f} - the whole file drifted from "
                    f"the checkpoint, not one part of it")
    else:
        log("PASS", f"values track the checkpoint across every role, allowing for what "
                    f"each tensor's quant type costs (median {med_all:.5f}, worst "
                    f"{worst[0][0]:.4f} on {worst[0][3]})")


In [ ]:
# @title check_companions · what this file needs wired up to run at all

# caption input width -> which text encoder produced it. Families that share tensor
# names can still take different text encoders, and the width is what tells them apart.
CAP_WIDTH = {
    2560: ("Qwen3-4B", "qwen_3_4b.safetensors"),
    2304: ("Gemma2-2B", "gemma_2_2b.safetensors"),
    4096: ("a 4096-wide encoder, T5-XXL or similar", "check the model card"),
    3584: ("Qwen2.5-VL-7B", "qwen_2.5_vl_7b.safetensors"),
}


def check_companions(gg, reader):
    """A diffusion gguf is the transformer and nothing else. Nothing in the file records
    which VAE and text encoder belong with it - but the width of the caption projection
    does pin down the encoder, and a mismatched one is the most common reason a file that
    passes every check still produces nothing usable."""
    cap = next((n for n in gg if n.endswith(("cap_embedder.1.weight",
                                             "cap_embedder.0.weight",
                                             "context_embedder.weight"))), None)
    if cap is None:
        log("INFO", "no caption projection here, so the text encoder can't be inferred")
        return

    width = gg[cap]["shape"][-1]
    name, file = CAP_WIDTH.get(width, (None, None))
    if name:
        log("INFO", f"caption features are {width} wide, which is {name} - the text "
                    f"encoder has to be {file}")
    else:
        log("INFO", f"caption features are {width} wide. Whatever text encoder has that "
                    f"hidden size is the one this needs.")

    arch = read_field(reader, "general.architecture")
    if arch == "lumina2":
        log("INFO", "loaded as 'lumina2': CLIPLoader type must be set to lumina2, and "
                    "the VAE is the 16-channel Flux autoencoder (ae.safetensors)")
    log("INFO", "a Turbo or distilled checkpoint wants CFG 1.0 and few steps - carrying "
                "a base model's CFG and step count across produces saturated, broken "
                "output from a file that is entirely correct")


In [ ]:
if not DO_CHECK:
    print("convert only, skipping the checks")
else:
    _current = "files in the folder"
    REPORT.setdefault(_current, [])
    print("=== files in the folder ===")
    check_files_exist()

    if SRC_BITS and SRC_BITS < 16:
        log("WARN", f"the source was {main_dtype}, only {SRC_BITS} bits per weight, so "
                    f"everything here carries a second rounding step")

    BASE_TENSORS = None
    if COMPARE_QUANTS_TO_BASE and BASE is not None:
        _, BASE_TENSORS = read_gguf(BASE)

    for path in FILES:
        _current = path.name
        _kind = "mmproj" if path in MMPROJ else "model"
        REPORT.setdefault(_current, [])
        print(f"\n=== {path.name}  ({path.stat().st_size / 2**30:.2f} GB) ===")

        reader, gg = read_gguf(path)
        if not gg:
            log("FAIL", "no tensors in this file")
            continue

        check_stamp(path)
        check_arch(reader)
        log("INFO", f"file_type {read_field(reader, 'general.file_type')}, {len(gg)} tensors")

        # llama.cpp renames tensors on the way in, so only the diffusion path matches by name
        if SRC_SHAPES and COMPARE_TO_SOURCE and MODE == "diffusion" and _kind == "model":
            check_source(gg)

        check_dtypes(gg)

        if _kind == "mmproj":
            check_mmproj(gg)
        elif MODE == "llm":
            check_tokenizer(reader, gg)

        if BASE_TENSORS is not None and path != BASE and _kind == "model":
            check_vs_base(gg, BASE_TENSORS)

        if NAN_SAMPLE is not None:
            check_nan(gg)

        if MODE == "diffusion" and _kind == "model":
            check_magnitude(gg, path)
            check_ops_coverage(gg)
            check_companions(gg, reader)
            if DIFF_SAMPLE and COMPARE_TO_SOURCE and SRC is not None and SRC_SHAPES:
                check_values(gg, path)

        del reader, gg

In [ ]:
# @title Verdict

fails = sum(1 for m in REPORT.values() for lvl, _ in m if lvl == "FAIL")
warns = sum(1 for m in REPORT.values() for lvl, _ in m if lvl == "WARN")
OK = fails == 0 and not (STRICT and warns)

print("=" * 62)
for name, msgs in REPORT.items():
    f = sum(1 for lvl, _ in msgs if lvl == "FAIL")
    w = sum(1 for lvl, _ in msgs if lvl == "WARN")
    print(f"  {'FAIL' if f else 'WARN' if w else 'ok  '}  {name:48} {f} fail  {w} warn")
print("=" * 62)
print(f"\n{fails} fail, {warns} warn  ->  {'good to ship' if OK else 'do not ship'}")

if not OK:
    said = " ".join(m for msgs in REPORT.values() for _, m in msgs)
    steps = []
    if "packed integer codes" in said:
        steps.append("The scales were never applied. Re-run 1f and read the `direction:` "
                     "line in the already-quantized cell - if it says neither direction "
                     "works, the scales aren't in this checkpoint and you need the float "
                     "weights instead.")
    if "no GGMLOps layer will dequantize" in said:
        steps.append("Add the listed names to KEEP_IN_F32 in 1e and convert again. Those "
                     "weights have to ship as F32 because nothing dequantizes them at "
                     "runtime.")
    if "out of line" in said or "far off the rest" in said:
        steps.append("One part of the network drifted while the rest converted cleanly, "
                     "so this is a conversion fault, not quantization loss. Set 1c to "
                     "build only the float base, run the checks on that alone, and see "
                     "whether the same role is still off - that separates a fault in "
                     "convert_diffusion from one in the quantize step.")
    if "must stay F32" in said or "single-row tensor" in said:
        steps.append("Weights that needed F32 were quantized. Check what the architecture "
                     "cell derived, and add anything missing to KEEP_IN_F32 in 1e.")
    if not steps:
        steps.append("Read the FAIL lines above in order - the first one is usually the "
                     "cause and the rest are consequences.")

    print("\nwhat to do next")
    for i, s in enumerate(steps, 1):
        print(f"  {i}. {s}")
    print("\n  In every case, the fastest way to halve the problem is to build the float")
    print("  base on its own in 1c and load that. If the float file is good, the fault is")
    print("  in quantization. If it shows the same thing, the fault is in conversion and")
    print("  nothing downstream of it is worth debugging yet.")

## Report and upload

In [ ]:
if SAVE_REPORT and REPORT:
    lines = [
        "# GGUF conversion report", "",
        f"- task: `{TASK}` ({MODE} / {SUBTASK})",
        f"- source: `{SRC_LABEL}`",
        f"- source id: `{SRC_FP.get('hash', 'not stamped')}` ({SRC_FP.get('method', 'off')})",
        f"- architecture: `{ARCH}`" + (" (derived)" if DERIVED else ""),
        f"- detected: {DETECTED or 'no known family'} via {ARCH_SOURCE}",
        f"- kept in F32: {len(F32_PATTERNS)} tensor(s)",
        f"- built: {', '.join(EXPECT)}",
        f"- result: **{'good to ship' if OK else 'do not ship'}** ({fails} fail, {warns} warn)",
        "",
    ]
    for name, msgs in REPORT.items():
        if not msgs:
            continue
        lines += [f"## {name}", ""]
        lines += [f"- **{lvl}** {msg}" if lvl in ("FAIL", "WARN") else f"- {lvl} {msg}"
                  for lvl, msg in msgs]
        lines.append("")

    report_path = OUT / "verification_report.md"
    report_path.write_text("\n".join(lines))
    print("wrote", report_path)

if UPLOAD_REPO.strip():
    if not OK:
        print("not uploading, the checks failed. Fix them or upload by hand.")
    else:
        from huggingface_hub import HfApi
        api = HfApi(token=HF_TOK or None)
        api.create_repo(UPLOAD_REPO.strip(), exist_ok=True)
        api.upload_folder(folder_path=str(OUT), repo_id=UPLOAD_REPO.strip())
        print("uploaded to", UPLOAD_REPO.strip())